# Setup and configuration

In [1]:
import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats

In [2]:
ROOT_DIR = Path.cwd()

DATA_DIR = ROOT_DIR / "data"
USER_DATA_DIR = DATA_DIR / "User Data"
USER_TRADES_DIR = DATA_DIR / "User Trades"

OUTPUT_DIR = ROOT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Raw file audit

## Expected columns

In [4]:
TRADER_EXPECTED_COLUMNS = {
    "account",
    "email",
    "ip_address",
    "ip address",
    "telegram_username",
    "challenge_type_id",
}

TRADE_EXPECTED_COLUMNS = {
    "accountid",
    "orderid",
    "opendatetime",
    "closedatetime",
    "profit",
    "reverseprofit",
    "netprofit",
    "commission",
    "swap",
    "amount",
    "openprice",
    "closeprice",
    "slprice",
    "tpprice",
    "side",
    "currency",
    "opentradecrossprice",
    "closetradecrossprice",
    "usergroupid",
    "campaignid",
}

## Audit function

In [5]:
def audit_file(
    path: str | Path,
    expected_columns: set[str],
    min_matches: int = 2,
) -> dict:
    """Scans a CSV or Excel file for embedded duplicate-header rows.

    The file is loaded without treating the first row as a header. Each
    subsequent row is checked for values that match known column names.
    Rows with at least `min_matches` matching values are flagged as possible
    embedded duplicate headers.

    Args:
        path: Path to the CSV or Excel file to audit.
        expected_columns: Normalized column names expected in the file.
        min_matches: Minimum number of column matches required to flag a row.
            Defaults to 2.

    Returns:
        A dictionary containing:
            - `file`: File name.
            - `total_rows`: Number of data rows excluding the true header.
            - `embedded_header_count`: Number of possible embedded-header rows.
            - `embedded_header_row_numbers`: Spreadsheet-style row numbers of
              the suspicious rows.
            - `error`: Error message if the file could not be read, otherwise
              `None`.
    """
    path = Path(path)

    try:
        if path.suffix.lower() == ".xlsx":
            raw = pd.read_excel(path, header=None)
        elif path.suffix.lower() == ".csv":
            raw = pd.read_csv(path, header=None)
        else:
            return {
                "file": path.name,
                "total_rows": None,
                "embedded_header_count": None,
                "embedded_header_row_numbers": [],
                "error": f"Unsupported file type: {path.suffix}",
            }

    except Exception as e:
        return {
            "file": path.name,
            "total_rows": None,
            "embedded_header_count": None,
            "embedded_header_row_numbers": [],
            "error": str(e),
        }

    total_rows = max(len(raw) - 1, 0)
    embedded_header_row_numbers = []

    for row_position in range(1, len(raw)):
        row_values = {
            str(value).strip().lower()
            for value in raw.iloc[row_position].tolist()
            if pd.notna(value)
        }

        n_matches = len(row_values.intersection(expected_columns))

        if n_matches >= min_matches:
            embedded_header_row_numbers.append(row_position + 1)

    return {
        "file": path.name,
        "total_rows": total_rows,
        "embedded_header_count": len(embedded_header_row_numbers),
        "embedded_header_row_numbers": embedded_header_row_numbers,
        "error": None,
    }

## Discover source files function

In [6]:
def discover_source_files(
    user_data_dir: str | Path,
    user_trades_dir: str | Path,
) -> pd.DataFrame:
    """Discovers trader and trade files from their respective directories.

    Args:
        user_data_dir: Directory containing account or participant files.
        user_trades_dir: Directory containing XAUUSD trade files.

    Returns:
        A DataFrame containing the file path, file name, file type,
        extension, and size of each discovered source file.

    Raises:
        FileNotFoundError: If either source directory does not exist.
    """
    user_data_dir = Path(user_data_dir)
    user_trades_dir = Path(user_trades_dir)

    if not user_data_dir.exists():
        raise FileNotFoundError(
            f"User Data directory was not found: "
            f"{user_data_dir.resolve()}"
        )

    if not user_trades_dir.exists():
        raise FileNotFoundError(
            f"User Trades directory was not found: "
            f"{user_trades_dir.resolve()}"
        )

    records = []

    folder_mapping = {
        "trader": user_data_dir,
        "trade": user_trades_dir,
    }

    for file_type, folder in folder_mapping.items():
        for path in folder.rglob("*"):
            if not path.is_file():
                continue

            if path.suffix.lower() not in {".csv", ".xlsx"}:
                continue

            records.append({
                "path": path,
                "file": path.name,
                "file_type": file_type,
                "extension": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            })

    if not records:
        return pd.DataFrame(
            columns=[
                "path",
                "file",
                "file_type",
                "extension",
                "size_bytes",
            ]
        )

    return (
        pd.DataFrame(records)
        .sort_values(["file_type", "file"])
        .reset_index(drop=True)
    )

In [7]:
source_files = discover_source_files(
    user_data_dir=USER_DATA_DIR,
    user_trades_dir=USER_TRADES_DIR,
)

print("Total source files:", len(source_files))
print()
print(source_files["file_type"].value_counts())

display(source_files.head())

Total source files: 68

file_type
trade     34
trader    34
Name: count, dtype: int64


,path,file,file_type,extension,size_bytes
0,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,trade,.csv,175399
1,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,Campaign 34 Data 03 Mar 2026 XAUUSD only (1D).csv,trade,.csv,219862
2,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,Campaign 35 Data 06 Mar 2026 XAUUSD only (1D).csv,trade,.csv,239141
3,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,Campaign 36 Data 09 Mar 2026 XAUUSD only (1D).csv,trade,.csv,339547
4,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,Campaign 37 Data 13 Mar 2026 XAUUSD only (1D).csv,trade,.csv,332721


## Audit all discovered files

In [8]:
audit_records = []

for file_record in source_files.itertuples(index=False):
    if file_record.file_type == "trader":
        expected_columns = TRADER_EXPECTED_COLUMNS
    else:
        expected_columns = TRADE_EXPECTED_COLUMNS

    audit_result = audit_file(
        path=file_record.path,
        expected_columns=expected_columns,
        min_matches=2,
    )

    audit_result["file_type"] = file_record.file_type
    audit_result["path"] = file_record.path

    audit_records.append(audit_result)

audit_df = pd.DataFrame(audit_records)

In [9]:
audit_summary = pd.Series({
    "files_audited": len(audit_df),
    "files_with_read_errors": audit_df["error"].notna().sum(),
    "files_with_embedded_headers": (
        audit_df["embedded_header_count"].fillna(0) > 0
    ).sum(),
    "embedded_header_rows_found": (
        audit_df["embedded_header_count"].fillna(0).sum()
    ),
})

display(audit_summary.to_frame("value"))

,value
files_audited,68
files_with_read_errors,0
files_with_embedded_headers,1
embedded_header_rows_found,4


In [10]:
audit_issues = audit_df.loc[
    audit_df["error"].notna()
    | audit_df["embedded_header_count"].fillna(0).gt(0),
    [
        "file_type",
        "file",
        "total_rows",
        "embedded_header_count",
        "embedded_header_row_numbers",
        "error",
    ],
]

display(audit_issues)

,file_type,file,total_rows,embedded_header_count,embedded_header_row_numbers,error
58,trader,Campaign 57 Data 02 June 2026 Traders only (1D...,504,4,"[102, 203, 304, 405]",None


In [11]:
audit_df.to_csv(
    OUTPUT_DIR / "raw_file_audit.csv",
    index=False,
)

# Load and standardize campaign files

## Define filename pattern

In [12]:
FILENAME_PATTERN = re.compile(
    r"Campaign[_\s]*(\d+)"
    r"[_\s]*Data[_\s]*"
    r"(\d{1,2}[_\s]*[A-Za-z]+[_\s]*\d{4})"
    r".*?"
    r"(Traders[\s_]*only|XAUUSD[\s_]*only)",
    re.IGNORECASE,
)

## Define parsing function

In [13]:
def parse_campaign_date(date_string: str) -> pd.Timestamp:
    """Parse a campaign date with an abbreviated or full month name.

    Args:
        date_string: Campaign date extracted from a filename.

    Returns:
        A parsed pandas Timestamp, or `pd.NaT` if the date cannot be
        parsed using a supported format.
    """
    normalized_date_string = re.sub(
        r"[_\s]+",
        " ",
        date_string.strip(),
    )

    supported_formats = [
        "%d %b %Y",  # Example: 02 Jun 2026
        "%d %B %Y",  # Example: 02 June 2026
    ]

    for date_format in supported_formats:
        parsed_date = pd.to_datetime(
            normalized_date_string,
            format=date_format,
            errors="coerce",
        )

        if pd.notna(parsed_date):
            return parsed_date

    return pd.NaT

In [14]:
def parse_filename(path: str | Path) -> dict:
    """Extract campaign metadata from a trader or trade filename.

    The function first attempts to match the complete expected filename
    pattern. If the complete pattern does not match, it applies fallback
    rules to extract the campaign ID and determine the file type.

    Args:
        path: Path to a trader-data or XAUUSD trade file.

    Returns:
        A dictionary containing:
            - `campaign_id`: Extracted campaign number, or `None`.
            - `campaign_date`: Parsed pandas Timestamp, or `pd.NaT`.
            - `file_type`: Either `"trader"`, `"trade"`, or `None`.
            - `filename_matched`: Whether the complete filename pattern
              matched.
            - `parse_warning`: Description of any parsing issue, otherwise
              `None`.
    """
    path = Path(path)
    filename = path.stem

    match = FILENAME_PATTERN.search(filename)

    if match:
        campaign_id = int(match.group(1))
        campaign_date = parse_campaign_date(match.group(2))

        raw_file_type = match.group(3).lower()
        file_type = (
            "trader"
            if "trader" in raw_file_type
            else "trade"
        )

        parse_warning = None

        if pd.isna(campaign_date):
            parse_warning = (
                "The campaign date could not be parsed from "
                f"'{match.group(2)}'."
            )

        return {
            "campaign_id": campaign_id,
            "campaign_date": campaign_date,
            "file_type": file_type,
            "filename_matched": True,
            "parse_warning": parse_warning,
        }

    campaign_match = re.search(
        r"Campaign[_\s]*(\d+)",
        filename,
        re.IGNORECASE,
    )

    campaign_id = (
        int(campaign_match.group(1))
        if campaign_match
        else None
    )

    filename_lower = filename.lower()
    parent_name_lower = path.parent.name.lower()

    if (
        "trader" in filename_lower
        or parent_name_lower == "user data"
    ):
        file_type = "trader"

    elif (
        "xauusd" in filename_lower
        or parent_name_lower == "user trades"
    ):
        file_type = "trade"

    else:
        file_type = None

    return {
        "campaign_id": campaign_id,
        "campaign_date": pd.NaT,
        "file_type": file_type,
        "filename_matched": False,
        "parse_warning": (
            "The filename did not match the complete expected pattern. "
            "Fallback extraction was applied."
        ),
    }

## Parse campaign metadata

In [15]:
filename_metadata_records = []

for file_record in source_files.itertuples(index=False):
    metadata = parse_filename(file_record.path)

    filename_metadata_records.append({
        "file": file_record.file,
        "path": file_record.path,
        "discovered_file_type": file_record.file_type,
        **metadata,
    })

filename_metadata_df = pd.DataFrame(
    filename_metadata_records
)

display(filename_metadata_df.head(10))

,file,path,discovered_file_type,campaign_id,campaign_date,file_type,filename_matched,parse_warning
0,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,33,2026-02-24,trade,True,None
1,Campaign 34 Data 03 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,34,2026-03-03,trade,True,None
2,Campaign 35 Data 06 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,35,2026-03-06,trade,True,None
3,Campaign 36 Data 09 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,36,2026-03-09,trade,True,None
4,Campaign 37 Data 13 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,37,2026-03-13,trade,True,None
5,Campaign 38 Data 17 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,38,2026-03-17,trade,True,None
6,Campaign 39 Data 20 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,39,2026-03-20,trade,True,None
7,Campaign 40 Data 24 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,40,2026-03-24,trade,True,None
8,Campaign 41 Data 27 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,41,2026-03-27,trade,True,None
9,Campaign 42 Data 31 Mar 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...,trade,42,2026-03-31,trade,True,None


In [16]:
parsing_summary = pd.Series({
    "total_files": len(filename_metadata_df),
    "complete_filename_matches": (
        filename_metadata_df["filename_matched"].sum()
    ),
    "fallback_matches": (
        ~filename_metadata_df["filename_matched"]
    ).sum(),
    "missing_campaign_ids": (
        filename_metadata_df["campaign_id"].isna().sum()
    ),
    "missing_campaign_dates": (
        filename_metadata_df["campaign_date"].isna().sum()
    ),
    "file_type_mismatches": (
        filename_metadata_df["discovered_file_type"]
        != filename_metadata_df["file_type"]
    ).sum(),
})

display(parsing_summary.to_frame("value"))

,value
total_files,68
complete_filename_matches,68
fallback_matches,0
missing_campaign_ids,0
missing_campaign_dates,0
file_type_mismatches,0


In [17]:
filename_parsing_issues = filename_metadata_df.loc[
    (~filename_metadata_df["filename_matched"])
    | filename_metadata_df["campaign_id"].isna()
    | filename_metadata_df["campaign_date"].isna()
    | (
        filename_metadata_df["discovered_file_type"]
        != filename_metadata_df["file_type"]
    ),
    [
        "file",
        "discovered_file_type",
        "campaign_id",
        "campaign_date",
        "file_type",
        "parse_warning",
    ],
]

display(filename_parsing_issues)

,file,discovered_file_type,campaign_id,campaign_date,file_type,parse_warning


In [18]:
campaign_file_counts = (
    filename_metadata_df
    .groupby(
        ["campaign_id", "file_type"],
        dropna=False,
    )
    .size()
    .unstack(
        fill_value=0,
    )
    .reset_index()
)

display(campaign_file_counts)

file_type,campaign_id,trade,trader
0,33,1,1
1,34,1,1
2,35,1,1
3,36,1,1
4,37,1,1
5,38,1,1
6,39,1,1
7,40,1,1
8,41,1,1
9,42,1,1


# Define column name normalization function

In [19]:
def normalize_column_name(column: object) -> str:
    """Converts a raw column name to lowercase snake case.

    Handles camel case, Pascal case, spaces, punctuation, and repeated
    underscores.

    Args:
        column: Original column name.

    Returns:
        The normalized snake-case column name.
    """
    column_name = str(column).strip()

    column_name = re.sub(
        r"([A-Z]+)([A-Z][a-z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"([a-z0-9])([A-Z])",
        r"\1_\2",
        column_name,
    )

    column_name = re.sub(
        r"[^A-Za-z0-9]+",
        "_",
        column_name,
    )

    column_name = re.sub(
        r"_+",
        "_",
        column_name,
    )

    return column_name.strip("_").lower()

In [20]:
print(normalize_column_name("IP ADDRESS"))
print(normalize_column_name("Telegram Username"))
print(normalize_column_name("accountId"))
print(normalize_column_name("openDateTime"))
print(normalize_column_name("netProfit"))

ip_address
telegram_username
account_id
open_date_time
net_profit


# Load and standardize trader files

## Define trader schema

In [21]:
TRADER_REQUIRED_COLUMNS = {
    "account_id",
    "email",
    "ip_address",
}

TRADER_OPTIONAL_COLUMNS = {
    "telegram_username",
    "challenge_type_id",
}

## Detect embedded trader headers

In [22]:
def identify_embedded_trader_headers(
    df: pd.DataFrame,
) -> pd.Series:
    """Identify embedded duplicate-header rows in a trader DataFrame.

    A row is identified as an embedded header when the account, email,
    or IP-address field repeats a recognized version of its column name.
    Missing values are treated as non-matches.

    Args:
        df: Standardized trader DataFrame.

    Returns:
        A Boolean Series where `True` identifies an embedded header row
        and `False` identifies a normal data row.
    """
    account_header_match = (
        df["account_id"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "account",
            "account_id",
            "account id",
            "accountid",
        })
        .fillna(False)
    )

    email_header_match = (
        df["email"]
        .astype("string")
        .str.strip()
        .str.lower()
        .eq("email")
        .fillna(False)
    )

    ip_address_header_match = (
        df["ip_address"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "ip_address",
            "ip address",
            "ipaddress",
        })
        .fillna(False)
    )

    embedded_header_mask = (
        account_header_match
        | email_header_match
        | ip_address_header_match
    )

    return embedded_header_mask.astype(bool)

## Define load_trader_file()

In [23]:
def load_trader_file(
    file_path: str | Path,
) -> tuple[pd.DataFrame, dict]:
    """Load and standardize one campaign trader file.

    The function reads a CSV or Excel trader file, normalizes its column
    names, validates the required schema, removes embedded duplicate-header
    rows, and attaches campaign metadata extracted from the filename.

    Args:
        file_path: Path to the trader CSV or Excel file.

    Returns:
        A tuple containing:
            - A standardized trader DataFrame.
            - A dictionary summarizing the loading and cleaning results.

    Raises:
        ValueError: If the file type is unsupported or required columns
            are missing.
    """
    file_path = Path(file_path)
    campaign_metadata = parse_filename(file_path)

    if file_path.suffix.lower() == ".xlsx":
        trader_df = pd.read_excel(file_path)

    elif file_path.suffix.lower() == ".csv":
        trader_df = pd.read_csv(file_path)

    else:
        raise ValueError(
            f"Unsupported trader file type: {file_path.suffix}"
        )

    source_row_count = len(trader_df)
    source_columns = trader_df.columns.tolist()

    # Preserve the original spreadsheet row number.
    trader_df.insert(
        loc=0,
        column="source_row_number",
        value=np.arange(2, len(trader_df) + 2),
    )

    # Standardize source column names.
    trader_df.columns = [
        normalize_column_name(column)
        for column in trader_df.columns
    ]

    trader_df = trader_df.rename(
        columns={
            "account": "account_id",
            "accountid": "account_id",
        }
    )

    missing_required_columns = (
        TRADER_REQUIRED_COLUMNS - set(trader_df.columns)
    )

    if missing_required_columns:
        raise ValueError(
            f"{file_path.name} is missing required trader columns: "
            f"{sorted(missing_required_columns)}"
        )

    # Add absent optional columns.
    for column in TRADER_OPTIONAL_COLUMNS:
        if column not in trader_df.columns:
            trader_df[column] = pd.NA

    embedded_header_mask = (
        identify_embedded_trader_headers(trader_df)
    )

    if embedded_header_mask.isna().any():
        raise RuntimeError(
            f"Embedded-header mask contains missing values for "
            f"{file_path.name}."
        )

    embedded_header_row_numbers = (
        trader_df.loc[
            embedded_header_mask,
            "source_row_number",
        ]
        .astype(int)
        .tolist()
    )

    trader_df = trader_df.loc[
        ~embedded_header_mask
    ].copy()

    removed_row_count = source_row_count - len(trader_df)

    if removed_row_count != len(embedded_header_row_numbers):
        raise RuntimeError(
            f"Row-removal mismatch in {file_path.name}: "
            f"{removed_row_count} rows were removed, but only "
            f"{len(embedded_header_row_numbers)} embedded-header "
            f"rows were recorded."
        )

    string_columns = [
        "account_id",
        "email",
        "ip_address",
        "telegram_username",
        "challenge_type_id",
    ]

    for column in string_columns:
        trader_df[column] = (
            trader_df[column]
            .astype("string")
            .str.strip()
            .replace({
                "": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "<NA>": pd.NA,
            })
        )

    # Attach campaign and source metadata.
    trader_df["campaign_id"] = (
        campaign_metadata["campaign_id"]
    )
    trader_df["campaign_date"] = (
        campaign_metadata["campaign_date"]
    )
    trader_df["source_file"] = file_path.name
    trader_df["source_path"] = str(file_path)

    standardized_columns = [
        "campaign_id",
        "campaign_date",
        "account_id",
        "email",
        "ip_address",
        "telegram_username",
        "challenge_type_id",
        "source_file",
        "source_path",
        "source_row_number",
    ]

    trader_df = (
        trader_df[standardized_columns]
        .reset_index(drop=True)
    )

    load_report = {
        "file_name": file_path.name,
        "campaign_id": campaign_metadata["campaign_id"],
        "campaign_date": campaign_metadata["campaign_date"],
        "source_row_count": source_row_count,
        "cleaned_row_count": len(trader_df),
        "removed_row_count": removed_row_count,
        "embedded_header_count": len(
            embedded_header_row_numbers
        ),
        "embedded_header_row_numbers": (
            embedded_header_row_numbers
        ),
        "source_columns": source_columns,
        "missing_account_id_count": int(
            trader_df["account_id"].isna().sum()
        ),
        "missing_email_count": int(
            trader_df["email"].isna().sum()
        ),
        "missing_ip_address_count": int(
            trader_df["ip_address"].isna().sum()
        ),
        "error": None,
    }

    return trader_df, load_report

## Test the first trader file

In [24]:
first_trader_path = source_files.loc[
    source_files["file_type"] == "trader",
    "path",
].iloc[0]

print(first_trader_path)

c:\Desktop\C22-veNTUre\data\User Data\Campaign 33 Data 24 Feb 2026 Traders only (1D).xlsx


In [25]:
test_trader, test_trader_report = load_trader_file(
    first_trader_path
)

In [26]:
print("Shape:", test_trader.shape)
print("Columns:", test_trader.columns.tolist())

display(test_trader.head())
display(pd.Series(test_trader_report).to_frame("value"))

Shape: (500, 10)
Columns: ['campaign_id', 'campaign_date', 'account_id', 'email', 'ip_address', 'telegram_username', 'challenge_type_id', 'source_file', 'source_path', 'source_row_number']


,campaign_id,campaign_date,account_id,email,ip_address,telegram_username,challenge_type_id,source_file,source_path,source_row_number
0,33,2026-02-24,D#1614601,jahanzaibkhalid154@gmail.com,182.185.152.215,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,c:\Desktop\C22-veNTUre\data\User Data\Campaign...,2
1,33,2026-02-24,D#1702708,muh.irwanto99@gmail.com,13.215.129.254,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,c:\Desktop\C22-veNTUre\data\User Data\Campaign...,3
2,33,2026-02-24,D#1702434,xbiswas598@gmail.com,77.111.246.27,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,c:\Desktop\C22-veNTUre\data\User Data\Campaign...,4
3,33,2026-02-24,D#1702729,Wachirajames817@gmail.com,102.210.40.114,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,c:\Desktop\C22-veNTUre\data\User Data\Campaign...,5
4,33,2026-02-24,D#1702577,naziahell988@gmail.com,37.111.148.168,<NA>,<NA>,Campaign 33 Data 24 Feb 2026 Traders only (1D)...,c:\Desktop\C22-veNTUre\data\User Data\Campaign...,6


,value
file_name,Campaign 33 Data 24 Feb 2026 Traders only (1D)...
campaign_id,33
campaign_date,2026-02-24 00:00:00
source_row_count,500
cleaned_row_count,500
removed_row_count,0
embedded_header_count,0
embedded_header_row_numbers,[]
source_columns,"[ACCOUNT, EMAIL, IP ADDRESS]"
missing_account_id_count,0


In [27]:
display(
    test_trader.dtypes
    .astype(str)
    .rename("dtype")
    .to_frame()
)

,dtype
campaign_id,int64
campaign_date,datetime64[us]
account_id,string
email,string
ip_address,string
telegram_username,string
challenge_type_id,string
source_file,str
source_path,str
source_row_number,int64


## Test campaign 57

In [28]:
campaign_57_trader_path = source_files.loc[
    (source_files["file_type"] == "trader")
    & source_files["file"].str.contains(
        r"Campaign[\s_]*57",
        case=False,
        regex=True,
    ),
    "path",
].iloc[0]

campaign_57_trader, campaign_57_report = (
    load_trader_file(campaign_57_trader_path)
)

display(
    pd.Series(campaign_57_report).to_frame("value")
)

,value
file_name,Campaign 57 Data 02 June 2026 Traders only (1D...
campaign_id,57
campaign_date,2026-06-02 00:00:00
source_row_count,504
cleaned_row_count,500
removed_row_count,4
embedded_header_count,4
embedded_header_row_numbers,"[102, 203, 304, 405]"
source_columns,"[ACCOUNT, EMAIL, IP ADDRESS]"
missing_account_id_count,0


In [29]:
print(
    "Number of embedded headers removed:",
    campaign_57_report[
        "embedded_header_count"
    ],
)

print(
    "Source row numbers:",
    campaign_57_report[
        "embedded_header_row_numbers"
    ],
)

Number of embedded headers removed: 4
Source row numbers: [102, 203, 304, 405]


# Load and standardize trade files

## Define trade schema

In [30]:
TRADE_ID_COLUMNS = [
    "account_id",
    "close_trade_id",
    "position_id",
    "close_order_id",
    "open_order_id",
    "user_group_id",
]

TRADE_DATETIME_COLUMNS = [
    "open_date_time",
    "close_date_time",
]

TRADE_NUMERIC_COLUMNS = [
    "lot_size",
    "duration_sec",
    "profit",
    "reverse_profit",
    "net_profit",
    "commission",
    "swap",
    "amount",
    "open_price",
    "close_price",
    "sl_price",
    "tp_price",
    "open_trade_cross_price",
    "close_trade_cross_price",
]

In [31]:
TRADE_COLUMN_ALIASES = {
    "account_id": "account_id",
    "accountid": "account_id",

    "close_trade_id": "close_trade_id",
    "closetradeid": "close_trade_id",

    "position_id": "position_id",
    "positionid": "position_id",

    "close_order_id": "close_order_id",
    "closeorderid": "close_order_id",

    "open_order_id": "open_order_id",
    "openorderid": "open_order_id",

    "user_group_id": "user_group_id",
    "usergroupid": "user_group_id",

    "open_date_time": "open_date_time",
    "opendatetime": "open_date_time",

    "close_date_time": "close_date_time",
    "closedatetime": "close_date_time",

    "lot_size": "lot_size",
    "lotsize": "lot_size",

    "duration_sec": "duration_sec",
    "durationsec": "duration_sec",

    "profit": "profit",

    "reverse_profit": "reverse_profit",
    "reverseprofit": "reverse_profit",

    "net_profit": "net_profit",
    "netprofit": "net_profit",

    "commission": "commission",
    "swap": "swap",
    "amount": "amount",

    "open_price": "open_price",
    "openprice": "open_price",

    "close_price": "close_price",
    "closeprice": "close_price",

    "sl_price": "sl_price",
    "slprice": "sl_price",

    "tp_price": "tp_price",
    "tpprice": "tp_price",

    "open_trade_cross_price": "open_trade_cross_price",
    "opentradecrossprice": "open_trade_cross_price",

    "close_trade_cross_price": "close_trade_cross_price",
    "closetradecrossprice": "close_trade_cross_price",

    "side": "side",
    "currency": "currency",

    "campaign_id": "source_campaign_id",
    "campaignid": "source_campaign_id",
}

## Define load_trade_file()

In [32]:
def load_trade_file(path: str | Path) -> tuple[pd.DataFrame, dict]:
    """Loads and standardizes one campaign trade file.

    Args:
        path: Path to a CSV or Excel trade file.

    Returns:
        A tuple containing:
            - A standardized trade DataFrame.
            - A dictionary containing loading and validation information.

    Raises:
        ValueError: If the file type is unsupported or required columns
            are absent.
    """
    path = Path(path)
    metadata = parse_filename(path)

    if path.suffix.lower() == ".csv":
        df = pd.read_csv(path, low_memory=False)
    elif path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        raise ValueError(
            f"Unsupported trade file type: {path.suffix}"
        )

    original_rows = len(df)
    original_columns = df.columns.tolist()

    normalized_columns = {
        column: normalize_column_name(column)
        for column in df.columns
    }

    df = df.rename(columns=normalized_columns)
    df = df.rename(columns=TRADE_COLUMN_ALIASES)

    df.insert(
        0,
        "source_row_number",
        np.arange(2, len(df) + 2),
    )

    required_columns = {
        "account_id",
        "open_date_time",
        "close_date_time",
        "amount",
        "net_profit",
    }

    missing_required = required_columns - set(df.columns)

    if missing_required:
        raise ValueError(
            f"{path.name} is missing required trade columns: "
            f"{sorted(missing_required)}"
        )

    # Remove actual embedded duplicate headers only.
    header_echo_mask = (
        df["account_id"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "account_id",
            "accountid",
            "account",
        })
        .fillna(False)
        |
        df["open_date_time"]
        .astype("string")
        .str.strip()
        .str.lower()
        .isin({
            "open_date_time",
            "opendatetime",
        })
        .fillna(False)
    ).astype(bool)

    embedded_header_rows = (
        df.loc[header_echo_mask, "source_row_number"]
        .astype(int)
        .tolist()
    )

    df = df.loc[~header_echo_mask].copy()

    rows_removed = original_rows - len(df)

    if rows_removed != len(embedded_header_rows):
        raise RuntimeError(
            f"Row-removal mismatch in {path.name}: "
            f"{rows_removed} rows removed but "
            f"{len(embedded_header_rows)} recorded."
        )

    for column in TRADE_ID_COLUMNS:
        if column in df.columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
                .replace({
                    "": pd.NA,
                    "nan": pd.NA,
                    "None": pd.NA,
                })
            )

    invalid_numeric_counts = {}

    for column in TRADE_NUMERIC_COLUMNS:
        if column not in df.columns:
            continue

        original_non_missing = df[column].notna()

        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

        invalid_numeric_counts[column] = int(
            (
                original_non_missing
                & df[column].isna()
            ).sum()
        )

    invalid_datetime_counts = {}

    for column in TRADE_DATETIME_COLUMNS:
        if column not in df.columns:
            continue

        original_non_missing = df[column].notna()

        df[column] = pd.to_datetime(
            df[column],
            errors="coerce",
            utc=True,
        )

        invalid_datetime_counts[column] = int(
            (
                original_non_missing
                & df[column].isna()
            ).sum()
        )

    if "side" in df.columns:
        df["side"] = (
            df["side"]
            .astype("string")
            .str.strip()
            .str.lower()
        )

    if "currency" in df.columns:
        df["currency"] = (
            df["currency"]
            .astype("string")
            .str.strip()
            .str.upper()
        )

    df["campaign_id"] = metadata["campaign_id"]
    df["campaign_date"] = metadata["campaign_date"]
    df["source_file"] = path.name
    df["source_path"] = str(path)

    loading_report = {
        "file": path.name,
        "campaign_id": metadata["campaign_id"],
        "campaign_date": metadata["campaign_date"],
        "original_rows": original_rows,
        "cleaned_rows": len(df),
        "rows_removed": rows_removed,
        "embedded_header_row_numbers": embedded_header_rows,
        "original_columns": original_columns,
        "invalid_numeric_counts": invalid_numeric_counts,
        "invalid_datetime_counts": invalid_datetime_counts,
        "missing_account_id": int(df["account_id"].isna().sum()),
        "missing_open_datetime": int(
            df["open_date_time"].isna().sum()
        ),
        "missing_close_datetime": int(
            df["close_date_time"].isna().sum()
        ),
        "error": None,
    }

    return df.reset_index(drop=True), loading_report

## Test the first trade file

In [33]:
first_trade_path = source_files.loc[
    source_files["file_type"] == "trade",
    "path",
].iloc[0]

test_trades, test_trade_report = load_trade_file(
    first_trade_path
)

print("Shape:", test_trades.shape)
print("Columns:", test_trades.columns.tolist())

display(test_trades.head())
display(pd.Series(test_trade_report).to_frame("value"))

Shape: (749, 30)
Columns: ['source_row_number', 'account_id', 'instrument', 'lot_size', 'close_trade_id', 'position_id', 'close_order_id', 'open_order_id', 'duration_sec', 'open_date_time', 'close_date_time', 'profit', 'reverse_profit', 'net_profit', 'commission', 'swap', 'amount', 'open_price', 'close_price', 'sl_price', 'tp_price', 'side', 'currency', 'open_trade_cross_price', 'close_trade_cross_price', 'user_group_id', 'campaign_id', 'campaign_date', 'source_file', 'source_path']


,source_row_number,account_id,instrument,lot_size,close_trade_id,position_id,close_order_id,open_order_id,duration_sec,open_date_time,...,tp_price,side,currency,open_trade_cross_price,close_trade_cross_price,user_group_id,campaign_id,campaign_date,source_file,source_path
0,2,D#1645625,XAUUSD,100,7349874591903930241,7349874591885628592,7349874591964660940,7349874591964660912,631,2026-02-24 00:49:38+00:00,...,5203.46,sell,USD,1,1,1583664,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...
1,3,D#1702379,XAUUSD,100,7349874591903930511,7349874591885628953,7349874591964663694,7349874591964663408,19,2026-02-24 01:02:19+00:00,...,NaN,buy,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...
2,4,D#1645651,XAUUSD,100,7349874591903930537,7349874591885628946,7349874591964663739,7349874591964663378,29,2026-02-24 01:02:15+00:00,...,NaN,sell,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...
3,5,D#1670922,XAUUSD,100,7349874591903930937,7349874591885629157,7349874591964665237,7349874591964665058,38,2026-02-24 01:07:03+00:00,...,NaN,sell,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...
4,6,D#1702617,XAUUSD,100,7349874591903931087,7349874591885629132,7349874591964665786,7349874591964664896,208,2026-02-24 01:06:23+00:00,...,NaN,buy,USD,1,1,1583663,33,2026-02-24,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv,c:\Desktop\C22-veNTUre\data\User Trades\Campai...


,value
file,Campaign 33 Data 24 Feb 2026 XAUUSD only (1D).csv
campaign_id,33
campaign_date,2026-02-24 00:00:00
original_rows,749
cleaned_rows,749
rows_removed,0
embedded_header_row_numbers,[]
original_columns,"[accountId, instrument, lotSize, closeTradeId,..."
invalid_numeric_counts,"{'lot_size': 0, 'duration_sec': 0, 'profit': 0..."
invalid_datetime_counts,"{'open_date_time': 0, 'close_date_time': 0}"


# Load all trader and trade files

## Load all trader files

In [34]:
trader_frames = []
trader_loading_reports = []

for file_record in source_files.loc[
    source_files["file_type"] == "trader"
].itertuples(index=False):

    try:
        file_df, report = load_trader_file(
            file_record.path
        )

        trader_frames.append(file_df)
        trader_loading_reports.append(report)

    except Exception as exc:
        trader_loading_reports.append({
            "file": file_record.file,
            "error": str(exc),
        })

trader = pd.concat(
    trader_frames,
    ignore_index=True,
)

trader_loading_report_df = pd.DataFrame(
    trader_loading_reports
)

## Load all trade files

In [35]:
trade_frames = []
trade_loading_reports = []

for file_record in source_files.loc[
    source_files["file_type"] == "trade"
].itertuples(index=False):

    try:
        file_df, report = load_trade_file(
            file_record.path
        )

        trade_frames.append(file_df)
        trade_loading_reports.append(report)

    except Exception as exc:
        trade_loading_reports.append({
            "file": file_record.file,
            "error": str(exc),
        })

trades = pd.concat(
    trade_frames,
    ignore_index=True,
)

trade_loading_report_df = pd.DataFrame(
    trade_loading_reports
)

In [36]:
trades = (
    trades
    .sort_values(
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "source_row_number",
        ]
    )
    .reset_index(drop=True)
)

trades["trade_row_id"] = np.arange(
    1,
    len(trades) + 1,
)

## Check loading errors

In [37]:
print("Trader loading errors:")

display(
    trader_loading_report_df.loc[
        trader_loading_report_df["error"].notna()
    ]
)

print("Trade loading errors:")

display(
    trade_loading_report_df.loc[
        trade_loading_report_df["error"].notna()
    ]
)

Trader loading errors:


,file_name,campaign_id,campaign_date,source_row_count,cleaned_row_count,removed_row_count,embedded_header_count,embedded_header_row_numbers,source_columns,missing_account_id_count,missing_email_count,missing_ip_address_count,error


Trade loading errors:


,file,campaign_id,campaign_date,original_rows,cleaned_rows,rows_removed,embedded_header_row_numbers,original_columns,invalid_numeric_counts,invalid_datetime_counts,missing_account_id,missing_open_datetime,missing_close_datetime,error


# Validate data

## Basic shape

In [38]:
data_summary = pd.Series({
    "trader_rows": len(trader),
    "trade_rows": len(trades),
    "trader_campaigns": trader[
        "campaign_id"
    ].nunique(),
    "trade_campaigns": trades[
        "campaign_id"
    ].nunique(),
    "registered_accounts": trader[
        "account_id"
    ].nunique(),
    "active_accounts": trades[
        "account_id"
    ].nunique(),
})

display(data_summary.to_frame("value"))

,value
trader_rows,15874
trade_rows,46520
trader_campaigns,34
trade_campaigns,34
registered_accounts,500
active_accounts,502


## Trader and trade account-campaign keys

In [39]:
trader_account_campaigns = (
    trader[
        ["campaign_id", "account_id"]
    ]
    .dropna(
        subset=["campaign_id", "account_id"]
    )
    .drop_duplicates()
)

active_account_campaigns = (
    trades[
        ["campaign_id", "account_id"]
    ]
    .dropna(
        subset=["campaign_id", "account_id"]
    )
    .drop_duplicates()
)

In [40]:
print(
    "Registered account-campaign combinations:",
    len(trader_account_campaigns),
)

print(
    "Active account-campaign combinations:",
    len(active_account_campaigns),
)

Registered account-campaign combinations: 15874
Active account-campaign combinations: 8165


## Find the campaigns where active exceeds registered

In [41]:
registered_by_campaign = (
    trader_account_campaigns
    .groupby("campaign_id")
    .size()
    .rename("registered_accounts")
)

active_by_campaign = (
    active_account_campaigns
    .groupby("campaign_id")
    .size()
    .rename("active_accounts")
)

trade_rows_by_campaign = (
    trades.groupby("campaign_id")
    .size()
    .rename("trade_rows")
)

campaign_summary = pd.concat(
    [
        registered_by_campaign,
        active_by_campaign,
        trade_rows_by_campaign,
    ],
    axis=1,
).reset_index()

In [42]:
count_columns = [
    "registered_accounts",
    "active_accounts",
    "trade_rows",
]

campaign_summary[count_columns] = (
    campaign_summary[count_columns]
    .fillna(0)
    .astype(int)
)

In [43]:
campaign_summary["participation_rate"] = np.where(
    campaign_summary["registered_accounts"] > 0,
    (
        campaign_summary["active_accounts"]
        / campaign_summary["registered_accounts"]
    ),
    np.nan,
)

In [44]:
campaign_summary["active_minus_registered"] = (
    campaign_summary["active_accounts"]
    - campaign_summary["registered_accounts"]
)

In [45]:
display(
    campaign_summary.sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered
0,33,500,137,749,0.274000,-363
1,34,281,192,942,0.683274,-89
2,35,500,185,1030,0.370000,-315
3,36,401,221,1457,0.551122,-180
4,37,455,236,1432,0.518681,-219
5,38,500,205,1271,0.410000,-295
6,39,340,205,1180,0.602941,-135
7,40,381,248,1432,0.650919,-133
8,41,223,177,1227,0.793722,-46
9,42,500,269,1674,0.538000,-231


## Check campaigns where active exceeds registered

In [46]:
campaigns_active_exceeds_registered = (
    campaign_summary.loc[
        campaign_summary["active_accounts"]
        > campaign_summary["registered_accounts"]
    ]
    .sort_values("campaign_id")
)

display(campaigns_active_exceeds_registered)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered


## Find the exact accounts missing from trader

In [47]:
active_trader_match = (
    active_account_campaigns
    .merge(
        trader_account_campaigns,
        on=["campaign_id", "account_id"],
        how="left",
        indicator=True,
    )
)

In [48]:
active_not_in_trader = (
    active_trader_match.loc[
        active_trader_match["_merge"] == "left_only",
        ["campaign_id", "account_id"],
    ]
    .sort_values(["campaign_id", "account_id"])
    .reset_index(drop=True)
)

print(
    "Active account-campaigns absent from trader:",
    len(active_not_in_trader),
)

display(active_not_in_trader)

Active account-campaigns absent from trader: 6


,campaign_id,account_id
0,33,D#1645625
1,33,D#1645639
2,33,D#1759507
3,34,D#1702452
4,37,D#1702495
5,37,D#1702657


In [49]:
active_not_in_trader_summary = (
    active_not_in_trader
    .groupby("campaign_id")
    .size()
    .rename("active_not_in_trader")
    .reset_index()
)

display(active_not_in_trader_summary)

,campaign_id,active_not_in_trader
0,33,3
1,34,1
2,37,2


In [50]:
campaign_summary = campaign_summary.merge(
    active_not_in_trader_summary,
    on="campaign_id",
    how="left",
)

campaign_summary["active_not_in_trader"] = (
    campaign_summary["active_not_in_trader"]
    .fillna(0)
    .astype(int)
)

display(
    campaign_summary.sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,trade_rows,participation_rate,active_minus_registered,active_not_in_trader
0,33,500,137,749,0.274000,-363,3
1,34,281,192,942,0.683274,-89,1
2,35,500,185,1030,0.370000,-315,0
3,36,401,221,1457,0.551122,-180,0
4,37,455,236,1432,0.518681,-219,2
5,38,500,205,1271,0.410000,-295,0
6,39,340,205,1180,0.602941,-135,0
7,40,381,248,1432,0.650919,-133,0
8,41,223,177,1227,0.793722,-46,0
9,42,500,269,1674,0.538000,-231,0


## Find registered accounts that never trade

In [51]:
trader_active_match = (
    trader_account_campaigns
    .merge(
        active_account_campaigns,
        on=["campaign_id", "account_id"],
        how="left",
        indicator=True,
    )
)

registered_not_active = (
    trader_active_match.loc[
        trader_active_match["_merge"] == "left_only",
        ["campaign_id", "account_id"],
    ]
    .sort_values(["campaign_id", "account_id"])
    .reset_index(drop=True)
)

print(
    "Registered account-campaigns with no trade:",
    len(registered_not_active),
)

display(registered_not_active.head(20))

Registered account-campaigns with no trade: 7715


,campaign_id,account_id
0,33,D#1589383
1,33,D#1589404
2,33,D#1589405
3,33,D#1589407
4,33,D#1589409
5,33,D#1589424
6,33,D#1589450
7,33,D#1589455
8,33,D#1589456
9,33,D#1589457


In [52]:
registered_not_active_summary = (
    registered_not_active
    .groupby("campaign_id")
    .size()
    .rename("registered_not_active")
    .reset_index()
)

campaign_summary = campaign_summary.merge(
    registered_not_active_summary,
    on="campaign_id",
    how="left",
)

campaign_summary["registered_not_active"] = (
    campaign_summary["registered_not_active"]
    .fillna(0)
    .astype(int)
)

In [53]:
display(
    campaign_summary[
        [
            "campaign_id",
            "registered_accounts",
            "active_accounts",
            "active_not_in_trader",
            "registered_not_active",
            "trade_rows",
            "participation_rate",
            "active_minus_registered",
        ]
    ].sort_values("campaign_id")
)

,campaign_id,registered_accounts,active_accounts,active_not_in_trader,registered_not_active,trade_rows,participation_rate,active_minus_registered
0,33,500,137,3,366,749,0.274000,-363
1,34,281,192,1,90,942,0.683274,-89
2,35,500,185,0,315,1030,0.370000,-315
3,36,401,221,0,180,1457,0.551122,-180
4,37,455,236,2,221,1432,0.518681,-219
5,38,500,205,0,295,1271,0.410000,-295
6,39,340,205,0,135,1180,0.602941,-135
7,40,381,248,0,133,1432,0.650919,-133
8,41,223,177,0,46,1227,0.793722,-46
9,42,500,269,0,231,1674,0.538000,-231


In [54]:
data_summary = pd.Series({
    "trader_rows": len(trader),
    "trade_rows": len(trades),
    "trader_campaigns": trader["campaign_id"].nunique(),
    "trade_campaigns": trades["campaign_id"].nunique(),
    "unique_registered_account_ids": (
        trader["account_id"].nunique()
    ),
    "unique_active_account_ids": (
        trades["account_id"].nunique()
    ),
    "registered_account_campaigns": len(
        trader_account_campaigns
    ),
    "active_account_campaigns": len(
        active_account_campaigns
    ),
    "active_account_campaigns_not_in_trader": len(
        active_not_in_trader
    ),
})

display(data_summary.to_frame("value"))

,value
trader_rows,15874
trade_rows,46520
trader_campaigns,34
trade_campaigns,34
unique_registered_account_ids,500
unique_active_account_ids,502
registered_account_campaigns,15874
active_account_campaigns,8165
active_account_campaigns_not_in_trader,6


## Find registered-only and active-only account IDs

In [55]:
registered_account_ids = set(
    trader["account_id"].dropna().unique()
)

active_account_ids = set(
    trades["account_id"].dropna().unique()
)

active_only_account_ids = sorted(
    active_account_ids - registered_account_ids
)

registered_only_account_ids = sorted(
    registered_account_ids - active_account_ids
)

print("Active-only account IDs:", active_only_account_ids)
print("Count:", len(active_only_account_ids))

print("\nRegistered-only account IDs:", registered_only_account_ids)
print("Count:", len(registered_only_account_ids))

Active-only account IDs: ['D#1645625', 'D#1645639', 'D#1759507']
Count: 3

Registered-only account IDs: ['D#1589383']
Count: 1


## Check trade duration

In [56]:
trades["calculated_duration_sec"] = (
    trades["close_date_time"]
    - trades["open_date_time"]
).dt.total_seconds()

In [57]:
duration_validation = pd.Series({
    "missing_open_datetime": trades[
        "open_date_time"
    ].isna().sum(),
    "missing_close_datetime": trades[
        "close_date_time"
    ].isna().sum(),
    "negative_duration": (
        trades["calculated_duration_sec"] < 0
    ).sum(),
    "zero_duration": (
        trades["calculated_duration_sec"] == 0
    ).sum(),
})

display(duration_validation.to_frame("value"))

,value
missing_open_datetime,0
missing_close_datetime,0
negative_duration,0
zero_duration,13


In [58]:
trades["duration_sec_comparison"] = (trades["duration_sec"] - trades["calculated_duration_sec"]).abs()
exactly_equal = (
    trades[
        "duration_sec_comparison"
    ]
    .eq(0)
    .all()
)

print(
    "All duration values are exactly equal:",
    exactly_equal,
)

All duration values are exactly equal: True


## Validate P&L reconciliation

In [59]:
required_pnl_columns = {
    "net_profit",
    "profit",
    "commission",
    "swap",
}

if required_pnl_columns.issubset(trades.columns):
    trades["expected_net_profit"] = (
        trades["profit"].fillna(0)
        + trades["commission"].fillna(0)
        + trades["swap"].fillna(0)
    )

    trades["net_profit_difference"] = (
        trades["net_profit"]
        - trades["expected_net_profit"]
    )

    trades["net_profit_reconciles"] = (
        trades["net_profit_difference"].abs()
        <= 0.01
    )

    print(
        "Rows failing P&L reconciliation:",
        (
            ~trades["net_profit_reconciles"]
        ).sum(),
    )

Rows failing P&L reconciliation: 0


## Check identifier duplication

In [60]:
identifier_checks = {}

for column in [
    "close_trade_id",
    "open_order_id",
    "close_order_id",
    "position_id",
]:
    if column in trades.columns:
        identifier_checks[
            f"duplicate_{column}"
        ] = trades.duplicated(
            subset=["campaign_id", column],
            keep=False,
        ).sum()

display(
    pd.Series(identifier_checks).to_frame("value")
)

,value
duplicate_close_trade_id,3138
duplicate_open_order_id,3138
duplicate_close_order_id,3138
duplicate_position_id,3138


# Previous completed trade

## Define attach previous completed trade function

In [61]:
def attach_previous_completed_trade(
    trades: pd.DataFrame,
) -> pd.DataFrame:
    """Attaches the most recently completed prior trade to each trade.

    Only trades whose close time occurred on or before the current trade's
    open time are eligible. This prevents unresolved or overlapping trades
    from leaking future outcomes into behavioral features.

    Args:
        trades: Standardized trade-level DataFrame.

    Returns:
        A copy of the trade DataFrame with previous-completed-trade
        information attached.
    """
    result_frames = []

    grouping_columns = [
        "campaign_id",
        "account_id",
    ]

    for _, group in trades.groupby(
        grouping_columns,
        dropna=False,
        sort=False,
    ):
        current = (
            group
            .sort_values(
                [
                    "open_date_time",
                    "close_date_time",
                    "trade_row_id",
                ]
            )
            .copy()
        )

        completed = (
            group.loc[
                group["close_date_time"].notna()
            ]
            .sort_values("close_date_time")
            .copy()
        )

        previous_columns = [
            "close_date_time",
            "net_profit",
            "amount",
            "position_id",
        ]

        previous_columns = [
            column
            for column in previous_columns
            if column in completed.columns
        ]

        previous = completed[
            previous_columns
        ].rename(
            columns={
                "close_date_time":
                    "previous_completed_close_date_time",
                "net_profit":
                    "previous_completed_net_profit",
                "amount":
                    "previous_completed_amount",
                "position_id":
                    "previous_completed_position_id",
            }
        )

        merged = pd.merge_asof(
            current.sort_values("open_date_time"),
            previous.sort_values(
                "previous_completed_close_date_time"
            ),
            left_on="open_date_time",
            right_on="previous_completed_close_date_time",
            direction="backward",
            allow_exact_matches=True,
        )

        result_frames.append(merged)

    result = pd.concat(
        result_frames,
        ignore_index=True,
    )

    result["previous_completed_was_loss"] = (
        result["previous_completed_net_profit"] < 0
    )

    result["previous_completed_was_win"] = (
        result["previous_completed_net_profit"] > 0
    )

    result["reentry_gap_minutes"] = (
        result["open_date_time"]
        - result[
            "previous_completed_close_date_time"
        ]
    ).dt.total_seconds() / 60

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [62]:
trades_with_previous_completed = (
    attach_previous_completed_trade(trades)
)

In [63]:
display(
    trades_with_previous_completed[
        [
            "campaign_id",
            "account_id",
            "open_date_time",
            "close_date_time",
            "net_profit",
            "previous_completed_close_date_time",
            "previous_completed_net_profit",
            "previous_completed_was_loss",
            "reentry_gap_minutes",
        ]
    ].head(20)
)

,campaign_id,account_id,open_date_time,close_date_time,net_profit,previous_completed_close_date_time,previous_completed_net_profit,previous_completed_was_loss,reentry_gap_minutes
0,33,D#1589378,2026-02-24 12:01:08+00:00,2026-02-24 12:02:17+00:00,0.01,NaT,NaN,False,NaN
1,33,D#1589408,2026-02-24 02:46:35+00:00,2026-02-24 02:52:47+00:00,-187.92,NaT,NaN,False,NaN
2,33,D#1589408,2026-02-24 02:53:40+00:00,2026-02-24 02:56:49+00:00,5.30,2026-02-24 02:52:47+00:00,-187.92,True,0.883333
3,33,D#1589408,2026-02-24 02:57:37+00:00,2026-02-24 02:58:40+00:00,-72.00,2026-02-24 02:56:49+00:00,5.30,False,0.800000
4,33,D#1589410,2026-02-25 00:13:04+00:00,2026-02-25 00:21:26+00:00,45.40,NaT,NaN,False,NaN
5,33,D#1589410,2026-02-25 00:23:39+00:00,2026-02-25 00:43:42+00:00,42.20,2026-02-25 00:21:26+00:00,45.40,False,2.216667
6,33,D#1589419,2026-02-24 05:32:22+00:00,2026-02-25 00:43:48+00:00,-20.11,NaT,NaN,False,NaN
7,33,D#1589428,2026-02-24 02:38:47+00:00,2026-02-24 02:44:09+00:00,13.00,NaT,NaN,False,NaN
8,33,D#1589428,2026-02-24 07:24:19+00:00,2026-02-24 07:29:48+00:00,108.29,2026-02-24 02:44:09+00:00,13.00,False,280.166667
9,33,D#1589428,2026-02-24 08:38:11+00:00,2026-02-24 12:17:20+00:00,-334.36,2026-02-24 07:29:48+00:00,108.29,False,68.383333


## Validate no look-ahead leakage

In [64]:
previous_trade_leakage = (
    trades_with_previous_completed[
        "previous_completed_close_date_time"
    ]
    >
    trades_with_previous_completed[
        "open_date_time"
    ]
)

print(
    "Previous completed trades closing after current entry:",
    previous_trade_leakage.fillna(False).sum(),
)

Previous completed trades closing after current entry: 0


## Verify gaps are not negative

In [65]:
print(
    "Negative re-entry gaps:",
    (
        trades_with_previous_completed[
            "reentry_gap_minutes"
        ] < 0
    ).sum(),
)

Negative re-entry gaps: 0


# Trade idea assignment

In [66]:
def assign_trade_ideas(
    trades: pd.DataFrame,
    maximum_gap_minutes: float = 3.0,
) -> pd.DataFrame:
    """Assigns standardized directional trade-idea identifiers.

    Trades are grouped by account, campaign, and side. Within each group,
    trades are ordered chronologically. A new idea starts when the next
    trade opens more than `maximum_gap_minutes` after the latest close
    time of all trades currently assigned to the idea.

    Overlapping trades and same-direction re-entries within the permitted
    gap remain part of the same idea.

    Args:
        trades: Standardized trade-level DataFrame.
        maximum_gap_minutes: Maximum gap, in minutes, allowed between the
            current idea's latest close and the next trade's opening.

    Returns:
        A chronologically sorted copy containing:
            - `idea_number`: Sequential idea number within each
              account-campaign-side group.
            - `idea_id`: Composite identifier formed from account,
              campaign, side, and idea number.

    Raises:
        ValueError: If required columns are missing, timestamps are
            invalid, or idea assignment fails.
    """
    required_columns = {
        "trade_row_id",
        "account_id",
        "campaign_id",
        "side",
        "open_date_time",
        "close_date_time",
    }

    missing_columns = (
        required_columns
        - set(trades.columns)
    )

    if missing_columns:
        raise ValueError(
            "Missing columns required for idea grouping: "
            f"{sorted(missing_columns)}"
        )

    if maximum_gap_minutes < 0:
        raise ValueError(
            "`maximum_gap_minutes` must be non-negative."
        )

    result = trades.copy()

    for column in [
        "open_date_time",
        "close_date_time",
    ]:
        result[column] = pd.to_datetime(
            result[column],
            errors="coerce",
        )

    if result[
        [
            "open_date_time",
            "close_date_time",
        ]
    ].isna().any().any():
        raise ValueError(
            "Missing or invalid trade timestamps detected."
        )

    if (
        result["close_date_time"]
        < result["open_date_time"]
    ).any():
        raise ValueError(
            "At least one trade closes before it opens."
        )

    group_columns = [
        "account_id",
        "campaign_id",
        "side",
    ]

    sort_columns = (
        group_columns
        + [
            "open_date_time",
            "close_date_time",
            "trade_row_id",
        ]
    )

    result = (
        result
        .sort_values(sort_columns)
        .copy()
    )

    idea_numbers = pd.Series(
        index=result.index,
        dtype="Int64",
    )

    for _, group in result.groupby(
        group_columns,
        sort=False,
        dropna=False,
    ):
        group = group.sort_values(
            [
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )

        current_idea_number = 1
        current_idea_latest_close = pd.NaT

        for row in group.itertuples():
            if pd.isna(
                current_idea_latest_close
            ):
                current_idea_latest_close = (
                    row.close_date_time
                )
            else:
                gap_minutes = (
                    row.open_date_time
                    - current_idea_latest_close
                ).total_seconds() / 60

                if (
                    gap_minutes
                    > maximum_gap_minutes
                ):
                    current_idea_number += 1
                    current_idea_latest_close = (
                        row.close_date_time
                    )
                else:
                    current_idea_latest_close = max(
                        current_idea_latest_close,
                        row.close_date_time,
                    )

            idea_numbers.loc[
                row.Index
            ] = current_idea_number

    result["idea_number"] = idea_numbers

    if result["idea_number"].isna().any():
        raise ValueError(
            "Some trades were not assigned an idea number."
        )

    result["idea_id"] = (
        result["account_id"].astype(str)
        + "_"
        + result["campaign_id"].astype(str)
        + "_"
        + result["side"].astype(str)
        + "_"
        + result["idea_number"].astype(str)
    )

    duplicate_idea_mapping = (
        result
        .groupby("idea_id")[
            group_columns
            + [
                "idea_number",
            ]
        ]
        .nunique(dropna=False)
        .gt(1)
        .any(axis=1)
    )

    if duplicate_idea_mapping.any():
        raise ValueError(
            "At least one idea ID maps to multiple groups."
        )

    return (
        result
        .sort_values(
            [
                "campaign_id",
                "account_id",
                "open_date_time",
                "close_date_time",
                "trade_row_id",
            ]
        )
        .reset_index(drop=True)
    )

In [67]:
trades_with_ideas = (
    assign_trade_ideas(
        trades,
        maximum_gap_minutes=3.0,
    )
)

In [68]:
print(
    "Original trade rows:",
    len(trades),
)

print(
    "Rows after idea assignment:",
    len(trades_with_ideas),
)

print(
    "Number of ideas:",
    trades_with_ideas[
        "idea_id"
    ].nunique(),
)

Original trade rows: 46520
Rows after idea assignment: 46520
Number of ideas: 33379


## Confirm every trade received an idea ID

In [69]:
missing_idea_ids = (
    trades_with_ideas[
        "idea_id"
    ].isna().sum()
)

print(
    "Trades without idea ID:",
    missing_idea_ids,
)

Trades without idea ID: 0


## Confirm no idea spans different accounts, campaigns or sides

In [70]:
idea_group_validation = (
    trades_with_ideas
    .groupby("idea_id")
    .agg(
        account_count=(
            "account_id",
            "nunique",
        ),
        campaign_count=(
            "campaign_id",
            "nunique",
        ),
        side_count=(
            "side",
            "nunique",
        ),
    )
)

invalid_idea_groups = idea_group_validation.loc[
    (idea_group_validation["account_count"] > 1)
    | (idea_group_validation["campaign_count"] > 1)
    | (idea_group_validation["side_count"] > 1)
]

print(
    "Invalid idea groups:",
    len(invalid_idea_groups),
)

display(invalid_idea_groups.head())

Invalid idea groups: 0


,account_count,campaign_count,side_count
idea_id,,,


## Build idea-size summary

In [71]:
idea_size_summary = (
    trades_with_ideas
    .groupby("idea_id")
    .size()
    .rename("number_of_trades")
    .reset_index()
)

display(
    idea_size_summary[
        "number_of_trades"
    ].describe()
)

count    33379.000000
mean         1.393691
std          1.059802
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         30.000000
Name: number_of_trades, dtype: float64

In [72]:
idea_size_counts = (
    idea_size_summary[
        "number_of_trades"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "trades_per_idea"
    )
    .reset_index(
        name="number_of_ideas"
    )
)

display(
    idea_size_counts.head(20)
)

,trades_per_idea,number_of_ideas
0,1,25952
1,2,4757
2,3,1435
3,4,574
4,5,296
5,6,137
6,7,77
7,8,42
8,9,32
9,10,25


## Calculate multi-trade idea rates

In [73]:
multi_trade_idea_count = (
    idea_size_summary[
        "number_of_trades"
    ].gt(1).sum()
)

multi_trade_idea_rate = (
    idea_size_summary[
        "number_of_trades"
    ].gt(1).mean()
)

trades_in_multi_trade_ideas = (
    idea_size_summary.loc[
        idea_size_summary[
            "number_of_trades"
        ] > 1,
        "number_of_trades",
    ].sum()
)

print(
    "Multi-trade ideas:",
    multi_trade_idea_count,
)

print(
    "Share of ideas with multiple trades:",
    f"{multi_trade_idea_rate:.2%}",
)

print(
    "Trades belonging to multi-trade ideas:",
    trades_in_multi_trade_ideas,
)

print(
    "Share of trades in multi-trade ideas:",
    f"{trades_in_multi_trade_ideas / len(trades):.2%}",
)

Multi-trade ideas: 7427
Share of ideas with multiple trades: 22.25%
Trades belonging to multi-trade ideas: 20568
Share of trades in multi-trade ideas: 44.21%


## Inspect the largest ideas

In [74]:
largest_ideas = (
    idea_size_summary
    .sort_values(
        "number_of_trades",
        ascending=False,
    )
    .head(20)
)

display(largest_ideas)

,idea_id,number_of_trades
32555,D#1702739_40_sell_1,30
32554,D#1702739_40_buy_2,29
17384,D#1702467_44_buy_1,23
17838,D#1702476_63_sell_6,21
6039,D#1670967_46_buy_1,19
12221,D#1702375_40_sell_5,18
27325,D#1702656_61_buy_1,18
21956,D#1702551_49_sell_2,17
10961,D#1702354_62_sell_7,17
18546,D#1702488_41_sell_1,17


In [75]:
example_idea_id = (
    largest_ideas.iloc[0]["idea_id"]
)

example_idea = (
    trades_with_ideas.loc[
        trades_with_ideas[
            "idea_id"
        ] == example_idea_id
    ]
    .sort_values("open_date_time")
)

display(
    example_idea[
        [
            "campaign_id",
            "account_id",
            "side",
            "open_date_time",
            "close_date_time",
            "amount",
            "open_price",
            "close_price",
            "net_profit",
            "idea_id",
        ]
    ]
)

,campaign_id,account_id,side,open_date_time,close_date_time,amount,open_price,close_price,net_profit,idea_id
9423,40,D#1702739,sell,2026-03-24 04:05:45+00:00,2026-03-24 04:06:54+00:00,0.20,4356.73,4353.54,61.20,D#1702739_40_sell_1
9425,40,D#1702739,sell,2026-03-24 04:07:50+00:00,2026-03-24 04:08:09+00:00,0.20,4352.25,4350.00,42.40,D#1702739_40_sell_1
9427,40,D#1702739,sell,2026-03-24 04:08:41+00:00,2026-03-24 04:09:08+00:00,0.20,4348.91,4351.33,-51.00,D#1702739_40_sell_1
9429,40,D#1702739,sell,2026-03-24 04:09:39+00:00,2026-03-24 04:14:18+00:00,0.20,4350.12,4350.48,-9.80,D#1702739_40_sell_1
9431,40,D#1702739,sell,2026-03-24 04:15:56+00:00,2026-03-24 04:16:28+00:00,0.11,4347.46,4345.38,21.44,D#1702739_40_sell_1
9432,40,D#1702739,sell,2026-03-24 04:16:37+00:00,2026-03-24 04:16:57+00:00,0.11,4347.71,4345.82,19.35,D#1702739_40_sell_1
9434,40,D#1702739,sell,2026-03-24 04:17:34+00:00,2026-03-24 04:18:27+00:00,0.20,4342.30,4340.66,30.20,D#1702739_40_sell_1
9436,40,D#1702739,sell,2026-03-24 04:19:03+00:00,2026-03-24 04:19:31+00:00,0.20,4340.23,4336.94,63.20,D#1702739_40_sell_1
9437,40,D#1702739,sell,2026-03-24 04:19:49+00:00,2026-03-24 04:20:32+00:00,0.20,4331.97,4329.24,52.00,D#1702739_40_sell_1
9439,40,D#1702739,sell,2026-03-24 04:20:52+00:00,2026-03-24 04:21:09+00:00,0.20,4331.75,4334.32,-54.00,D#1702739_40_sell_1


## Calculate idea-level duration

In [76]:
idea_timing_summary = (
    trades_with_ideas
    .groupby("idea_id")
    .agg(
        idea_start_time=(
            "open_date_time",
            "min",
        ),
        idea_end_time=(
            "close_date_time",
            "max",
        ),
        number_of_trades=(
            "idea_id",
            "size",
        ),
    )
    .reset_index()
)

idea_timing_summary[
    "idea_duration_minutes"
] = (
    idea_timing_summary["idea_end_time"]
    - idea_timing_summary["idea_start_time"]
).dt.total_seconds() / 60

In [77]:
display(
    idea_timing_summary[
        "idea_duration_minutes"
    ].describe(
        percentiles=[
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
)

count    33379.000000
mean        33.235520
std         77.807406
min          0.000000
50%          8.983333
75%         27.491667
90%         79.283333
95%        148.251667
99%        398.529000
max       1374.933333
Name: idea_duration_minutes, dtype: float64

## Inspect the longest ideas

In [78]:
display(
    idea_timing_summary
    .sort_values(
        "idea_duration_minutes",
        ascending=False,
    )
    .head(20)
)

,idea_id,idea_start_time,idea_end_time,number_of_trades,idea_duration_minutes
25420,D#1702623_33_buy_1,2026-02-24 02:26:01+00:00,2026-02-25 01:20:57+00:00,2,1374.933333
5560,D#1670955_55_buy_1,2026-05-26 03:06:35+00:00,2026-05-27 00:55:31+00:00,1,1308.933333
19330,D#1702502_65_sell_1,2026-06-30 03:56:05+00:00,2026-07-01 00:58:01+00:00,2,1261.933333
17007,D#1702460_50_sell_1,2026-05-07 01:10:27+00:00,2026-05-07 22:02:18+00:00,1,1251.850000
12380,D#1702379_33_buy_2,2026-02-24 04:17:15+00:00,2026-02-25 01:01:27+00:00,1,1244.200000
5403,D#1670951_38_buy_2,2026-03-17 04:55:34+00:00,2026-03-18 00:47:36+00:00,1,1192.033333
15750,D#1702441_40_sell_1,2026-03-24 01:04:46+00:00,2026-03-24 20:25:01+00:00,1,1160.250000
456,D#1589419_33_buy_1,2026-02-24 05:32:22+00:00,2026-02-25 00:43:48+00:00,1,1151.433333
23410,D#1702577_33_sell_1,2026-02-24 06:42:15+00:00,2026-02-25 01:18:02+00:00,1,1115.783333
704,D#1589450_45_buy_1,2026-04-14 07:05:52+00:00,2026-04-15 00:50:37+00:00,1,1064.750000


## Check the actual gap within each idea

In [79]:
idea_gap_check = (
    trades_with_ideas
    .sort_values(
        [
            "idea_id",
            "open_date_time",
        ]
    )
    .copy()
)

idea_gap_check[
    "previous_trade_close_time"
] = (
    idea_gap_check
    .groupby("idea_id")[
        "close_date_time"
    ]
    .shift(1)
)

idea_gap_check[
    "gap_from_previous_close_minutes"
] = (
    idea_gap_check["open_date_time"]
    - idea_gap_check[
        "previous_trade_close_time"
    ]
).dt.total_seconds() / 60

In [80]:
display(
    idea_gap_check[
        "gap_from_previous_close_minutes"
    ].describe()
)

count    13141.000000
mean         0.882681
std          4.886909
min       -538.300000
25%          0.250000
50%          0.650000
75%          1.450000
max          3.000000
Name: gap_from_previous_close_minutes, dtype: float64

## Build idea-level features

In [81]:
trades_with_ideas = (
    trades_with_ideas
    .sort_values(
        [
            "idea_id",
            "open_date_time",
            "close_date_time",
        ]
    )
    .copy()
)

trades_with_ideas[
    "trade_number_within_idea"
] = (
    trades_with_ideas
    .groupby("idea_id")
    .cumcount()
    + 1
)

In [82]:
idea_features = (
    trades_with_ideas
    .groupby("idea_id", as_index=False)
    .agg(
        account_id=("account_id", "first"),
        campaign_id=("campaign_id", "first"),
        instrument=("instrument", "first"),
        side=("side", "first"),

        idea_start_time=("open_date_time", "min"),
        idea_end_time=("close_date_time", "max"),

        number_of_trades=("idea_id", "size"),

        total_amount=("amount", "sum"),
        mean_amount=("amount", "mean"),
        maximum_amount=("amount", "max"),
        minimum_amount=("amount", "min"),

        total_net_profit=("net_profit", "sum"),
        mean_net_profit=("net_profit", "mean"),
        maximum_net_profit=("net_profit", "max"),
        minimum_net_profit=("net_profit", "min"),
    )
)

idea_features["amount_range"] = (
    idea_features["maximum_amount"]
    - idea_features["minimum_amount"]
)

## Calculate duration

In [83]:
idea_features["idea_duration_minutes"] = (
    idea_features["idea_end_time"]
    - idea_features["idea_start_time"]
).dt.total_seconds() / 60

## Calculate profitability

In [84]:
idea_features["is_profitable_idea"] = (
    idea_features["total_net_profit"] > 0
)

idea_features["is_losing_idea"] = (
    idea_features["total_net_profit"] < 0
)

idea_features["is_breakeven_idea"] = (
    idea_features["total_net_profit"] == 0
)

In [85]:
print("Idea rows:", len(idea_features))
print("Unique ideas:", idea_features["idea_id"].nunique())

display(idea_features.head(10))

Idea rows: 33379
Unique ideas: 33379


,idea_id,account_id,campaign_id,instrument,side,idea_start_time,idea_end_time,number_of_trades,total_amount,mean_amount,...,minimum_amount,total_net_profit,mean_net_profit,maximum_net_profit,minimum_net_profit,amount_range,idea_duration_minutes,is_profitable_idea,is_losing_idea,is_breakeven_idea
0,D#1589378_33_sell_1,D#1589378,33,XAUUSD,sell,2026-02-24 12:01:08+00:00,2026-02-24 12:02:17+00:00,1,0.01,0.01,...,0.01,0.01,0.01,0.01,0.01,0.0,1.150000,True,False,False
1,D#1589378_37_buy_1,D#1589378,37,XAUUSD,buy,2026-03-13 03:48:36+00:00,2026-03-13 03:49:32+00:00,1,0.30,0.30,...,0.30,39.90,39.90,39.90,39.90,0.0,0.933333,True,False,False
2,D#1589378_37_buy_2,D#1589378,37,XAUUSD,buy,2026-03-13 03:52:48+00:00,2026-03-13 03:59:07+00:00,1,0.30,0.30,...,0.30,47.70,47.70,47.70,47.70,0.0,6.316667,True,False,False
3,D#1589378_37_buy_3,D#1589378,37,XAUUSD,buy,2026-03-13 06:24:41+00:00,2026-03-13 06:25:58+00:00,1,0.30,0.30,...,0.30,-72.90,-72.90,-72.90,-72.90,0.0,1.283333,False,True,False
4,D#1589378_37_sell_1,D#1589378,37,XAUUSD,sell,2026-03-13 02:18:44+00:00,2026-03-13 02:22:52+00:00,1,0.40,0.40,...,0.40,207.20,207.20,207.20,207.20,0.0,4.133333,True,False,False
5,D#1589378_37_sell_2,D#1589378,37,XAUUSD,sell,2026-03-13 04:10:03+00:00,2026-03-13 04:12:17+00:00,1,0.30,0.30,...,0.30,32.10,32.10,32.10,32.10,0.0,2.233333,True,False,False
6,D#1589378_37_sell_3,D#1589378,37,XAUUSD,sell,2026-03-13 04:26:54+00:00,2026-03-13 04:33:56+00:00,1,0.30,0.30,...,0.30,-27.30,-27.30,-27.30,-27.30,0.0,7.033333,False,True,False
7,D#1589378_37_sell_4,D#1589378,37,XAUUSD,sell,2026-03-13 04:47:32+00:00,2026-03-13 06:28:21+00:00,7,2.10,0.30,...,0.30,176.40,25.20,292.50,-218.70,0.0,100.816667,True,False,False
8,D#1589378_42_buy_1,D#1589378,42,XAUUSD,buy,2026-03-31 14:28:40+00:00,2026-03-31 14:33:40+00:00,1,0.10,0.10,...,0.10,96.30,96.30,96.30,96.30,0.0,5.000000,True,False,False
9,D#1589378_42_buy_2,D#1589378,42,XAUUSD,buy,2026-03-31 14:56:28+00:00,2026-03-31 14:58:45+00:00,1,0.05,0.05,...,0.05,5.34,5.34,5.34,5.34,0.0,2.283333,True,False,False


## Initial trade features

In [86]:
initial_trade_features = (
    trades_with_ideas.loc[
        trades_with_ideas[
            "trade_number_within_idea"
        ] == 1,
        [
            "idea_id",
            "amount",
            "open_date_time",
        ],
    ]
    .rename(
        columns={
            "amount": "initial_entry_amount",
            "open_date_time": "initial_entry_open_time",
        }
    )
)

In [87]:
idea_features = idea_features.merge(
    initial_trade_features,
    on="idea_id",
    how="left",
    validate="one_to_one",
)

In [88]:
idea_features["maximum_to_initial_entry_ratio"] = np.where(
    idea_features["initial_entry_amount"] > 0,
    (
        idea_features["maximum_amount"]
        / idea_features["initial_entry_amount"]
    ),
    np.nan,
)

In [89]:
idea_features["initial_entry_share"] = np.where(
    idea_features["initial_entry_amount"] > 0,
    (
        idea_features["initial_entry_amount"]
        / idea_features["total_amount"]
    ),
    np.nan,
)

In [90]:
idea_features["increased_position_size"] = (
    idea_features["maximum_amount"]
    > idea_features["initial_entry_amount"]
)

## Mark single and multi-trade ideas

In [91]:
idea_features["multi_trade_idea"] = (
    idea_features["number_of_trades"] > 1
)

In [92]:
display(
    idea_features["multi_trade_idea"]
    .value_counts()
    .rename_axis("multi_trade_idea")
    .reset_index(name="number_of_ideas")
)

,multi_trade_idea,number_of_ideas
0,False,25952
1,True,7427


## Calculate time to peak trade size

In [93]:
maximum_amount_rows = (
    trades_with_ideas
    .sort_values(
        [
            "idea_id",
            "open_date_time",
            "close_date_time",
        ]
    )
    .loc[
        lambda df: (
            df["amount"]
            == df.groupby("idea_id")["amount"]
            .transform("max")
        )
    ]
    .drop_duplicates(
        subset=["idea_id"],
        keep="first",
    )
    [
        [
            "idea_id",
            "open_date_time",
            "trade_number_within_idea",
        ]
    ]
    .rename(
        columns={
            "open_date_time": "peak_trade_open_time",
            "trade_number_within_idea": "peak_trade_number",
        }
    )
)

In [94]:
print(
    "Peak-size rows:",
    len(maximum_amount_rows),
)

print(
    "Unique peak-size ideas:",
    maximum_amount_rows["idea_id"].nunique(),
)

Peak-size rows: 33379
Unique peak-size ideas: 33379


In [95]:
idea_features = idea_features.merge(
    maximum_amount_rows,
    on="idea_id",
    how="left",
    validate="one_to_one",
)

In [96]:
idea_features["minutes_to_peak_trade_size"] = (
    idea_features["peak_trade_open_time"]
    - idea_features["idea_start_time"]
).dt.total_seconds() / 60

In [97]:
idea_features["peak_size_timing_ratio"] = np.where(
    idea_features["idea_duration_minutes"] > 0,
    (
        idea_features["minutes_to_peak_trade_size"]
        / idea_features["idea_duration_minutes"]
    ),
    np.nan,
)

In [98]:
display(
    idea_features.loc[
        idea_features["multi_trade_idea"],
        [
            "minutes_to_peak_trade_size",
            "peak_size_timing_ratio",
        ],
    ].describe()
)

,minutes_to_peak_trade_size,peak_size_timing_ratio
count,7427.000000,7427.000000
mean,7.262825,0.150212
std,36.854457,0.283320
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.941667,0.139188
max,835.366667,0.999343


## Measure each size transition

In [99]:
trades_with_ideas = (
    trades_with_ideas
    .sort_values(
        [
            "idea_id",
            "trade_number_within_idea",
        ]
    )
    .copy()
)

trades_with_ideas[
    "previous_amount_within_idea"
] = (
    trades_with_ideas
    .groupby("idea_id")["amount"]
    .shift(1)
)

In [100]:
trades_with_ideas[
    "amount_increased_from_previous"
] = (
    trades_with_ideas["amount"]
    >
    trades_with_ideas[
        "previous_amount_within_idea"
    ]
)

In [101]:
size_transition_features = (
    trades_with_ideas
    .groupby("idea_id", as_index=False)
    .agg(
        eligible_size_transitions=(
            "previous_amount_within_idea",
            lambda values: values.notna().sum(),
        ),
        size_increase_count=(
            "amount_increased_from_previous",
            "sum",
        ),
    )
)

In [102]:
size_transition_features["size_increase_rate"] = np.where(
    size_transition_features["eligible_size_transitions"] > 0,
    (
        size_transition_features["size_increase_count"]
        / size_transition_features["eligible_size_transitions"]
    ),
    np.nan,
)

In [103]:
idea_features = idea_features.merge(
    size_transition_features,
    on="idea_id",
    how="left",
    validate="one_to_one",
)

## Merge with ideas_with_trades

In [104]:
idea_feature_columns = [
    "idea_id",

    # Idea timing
    "idea_start_time",
    "idea_end_time",
    "idea_duration_minutes",

    # Number of trades
    "number_of_trades",
    "multi_trade_idea",

    # Amount features
    "total_amount",
    "mean_amount",
    "maximum_amount",
    "minimum_amount",
    "amount_range",

    # Initial-entry features
    "initial_entry_amount",
    "initial_entry_open_time",
    "maximum_to_initial_entry_ratio",
    "initial_entry_share",
    "increased_position_size",

    # Maximum-entry timing features
    "peak_trade_open_time",
    "peak_trade_number",
    "minutes_to_peak_trade_size",
    "peak_size_timing_ratio",

    # Sequential size-change features
    "eligible_size_transitions",
    "size_increase_count",
    "size_increase_rate",

    # Profit features
    "total_net_profit",
    "mean_net_profit",
    "maximum_net_profit",
    "minimum_net_profit",

    # Idea outcome labels
    "is_profitable_idea",
    "is_losing_idea",
    "is_breakeven_idea",
]

In [105]:
# Preserve the number of trade rows before merging.
trade_row_count_before_merge = len(trades_with_ideas)


# Make the cell safe to rerun by removing previously merged idea features.
existing_idea_feature_columns = [
    column
    for column in idea_feature_columns
    if column != "idea_id"
    and column in trades_with_ideas.columns
]

trades_with_ideas = trades_with_ideas.drop(
    columns=existing_idea_feature_columns
)


# Attach the idea-level features to every trade in that idea.
trades_with_ideas = trades_with_ideas.merge(
    idea_features[idea_feature_columns],
    on="idea_id",
    how="left",
    validate="many_to_one",
)

In [106]:
# Validation checks.
assert idea_features["idea_id"].is_unique

assert len(trades_with_ideas) == trade_row_count_before_merge

assert trades_with_ideas["idea_id"].notna().all()

assert (
    trades_with_ideas["number_of_trades"].notna().all()
)


print(
    "Idea-level feature rows:",
    len(idea_features),
)

print(
    "Trade rows after feature merge:",
    len(trades_with_ideas),
)

print(
    "Idea features attached:",
    len(idea_feature_columns) - 1,
)

display(idea_features.head())

display(trades_with_ideas.head())

Idea-level feature rows: 33379
Trade rows after feature merge: 46520
Idea features attached: 29


,idea_id,account_id,campaign_id,instrument,side,idea_start_time,idea_end_time,number_of_trades,total_amount,mean_amount,...,initial_entry_share,increased_position_size,multi_trade_idea,peak_trade_open_time,peak_trade_number,minutes_to_peak_trade_size,peak_size_timing_ratio,eligible_size_transitions,size_increase_count,size_increase_rate
0,D#1589378_33_sell_1,D#1589378,33,XAUUSD,sell,2026-02-24 12:01:08+00:00,2026-02-24 12:02:17+00:00,1,0.01,0.01,...,1.0,False,False,2026-02-24 12:01:08+00:00,1,0.0,0.0,0,0,NaN
1,D#1589378_37_buy_1,D#1589378,37,XAUUSD,buy,2026-03-13 03:48:36+00:00,2026-03-13 03:49:32+00:00,1,0.30,0.30,...,1.0,False,False,2026-03-13 03:48:36+00:00,1,0.0,0.0,0,0,NaN
2,D#1589378_37_buy_2,D#1589378,37,XAUUSD,buy,2026-03-13 03:52:48+00:00,2026-03-13 03:59:07+00:00,1,0.30,0.30,...,1.0,False,False,2026-03-13 03:52:48+00:00,1,0.0,0.0,0,0,NaN
3,D#1589378_37_buy_3,D#1589378,37,XAUUSD,buy,2026-03-13 06:24:41+00:00,2026-03-13 06:25:58+00:00,1,0.30,0.30,...,1.0,False,False,2026-03-13 06:24:41+00:00,1,0.0,0.0,0,0,NaN
4,D#1589378_37_sell_1,D#1589378,37,XAUUSD,sell,2026-03-13 02:18:44+00:00,2026-03-13 02:22:52+00:00,1,0.40,0.40,...,1.0,False,False,2026-03-13 02:18:44+00:00,1,0.0,0.0,0,0,NaN


,source_row_number,account_id,instrument,lot_size,close_trade_id,position_id,close_order_id,open_order_id,duration_sec,open_date_time,...,eligible_size_transitions,size_increase_count,size_increase_rate,total_net_profit,mean_net_profit,maximum_net_profit,minimum_net_profit,is_profitable_idea,is_losing_idea,is_breakeven_idea
0,469,D#1589378,XAUUSD,100,7349874591903961172,7349874591885644259,7349874591964763954,7349874591964763848,69,2026-02-24 12:01:08+00:00,...,0,0,NaN,0.01,0.01,0.01,0.01,True,False,False
1,230,D#1589378,XAUUSD,100,7349874591904982953,7349874591886146549,7349874591967228742,7349874591967228658,56,2026-03-13 03:48:36+00:00,...,0,0,NaN,39.90,39.90,39.90,39.90,True,False,False
2,238,D#1589378,XAUUSD,100,7349874591904983212,7349874591886146626,7349874591967229424,7349874591967228994,379,2026-03-13 03:52:48+00:00,...,0,0,NaN,47.70,47.70,47.70,47.70,True,False,False
3,469,D#1589378,XAUUSD,100,7349874591904988704,7349874591886149367,7349874591967241250,7349874591967241055,77,2026-03-13 06:24:41+00:00,...,0,0,NaN,-72.90,-72.90,-72.90,-72.90,False,True,False
4,110,D#1589378,XAUUSD,100,7349874591904979965,7349874591886145041,7349874591967221780,7349874591967221403,248,2026-03-13 02:18:44+00:00,...,0,0,NaN,207.20,207.20,207.20,207.20,True,False,False


# Create reusable master outcomes table

In [107]:
account_campaign_outcomes = (
    idea_features
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        account_campaign_net_profit=(
            "total_net_profit",
            "sum",
        ),
        number_of_ideas=(
            "idea_id",
            "nunique",
        ),
    )
)

account_campaign_outcomes["profitable"] = (
    account_campaign_outcomes[
        "account_campaign_net_profit"
    ] > 0
)

account_campaign_outcomes["profitability_group"] = np.where(
    account_campaign_outcomes["profitable"],
    "Profitable",
    "Unprofitable",
)

# Define Mann Whitney U Test function

In [108]:
def run_mann_whitney_comparison(
    data: pd.DataFrame,
    feature: str,
    group_column: str = "profitable",
) -> dict:
    """Compare a numeric feature between profitable and unprofitable groups.

    A positive rank-biserial correlation means that the feature tends
    to be higher among profitable account-campaigns.

    A negative rank-biserial correlation means that the feature tends
    to be higher among unprofitable account-campaigns.

    Args:
        data:
            Account-campaign-level DataFrame.

        feature:
            Name of the numeric feature being compared.

        group_column:
            Boolean column separating profitable and unprofitable
            account-campaigns.

    Returns:
        Sample sizes, medians, median difference, Mann–Whitney U
        statistic, p-value, and rank-biserial correlation.
    """
    required_columns = {
        feature,
        group_column,
    }

    missing_columns = required_columns.difference(data.columns)

    if missing_columns:
        raise KeyError(
            "Missing required columns: "
            f"{sorted(missing_columns)}"
        )

    if not pd.api.types.is_bool_dtype(
        data[group_column]
    ):
        raise TypeError(
            f"{group_column!r} must be a Boolean column."
        )

    numeric_feature = (
        pd.to_numeric(
            data[feature],
            errors="coerce",
        )
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
    )

    profitable_values = (
        numeric_feature.loc[
            data[group_column]
        ]
        .dropna()
    )

    unprofitable_values = (
        numeric_feature.loc[
            ~data[group_column]
        ]
        .dropna()
    )

    profitable_n = len(profitable_values)
    unprofitable_n = len(unprofitable_values)

    if profitable_n < 2 or unprofitable_n < 2:
        return {
            "feature": feature,
            "profitable_n": profitable_n,
            "unprofitable_n": unprofitable_n,
            "profitable_median": np.nan,
            "unprofitable_median": np.nan,
            "median_difference": np.nan,
            "u_statistic": np.nan,
            "p_value": np.nan,
            "rank_biserial": np.nan,
        }

    test_result = stats.mannwhitneyu(
        profitable_values,
        unprofitable_values,
        alternative="two-sided",
        method="auto",
    )

    profitable_median = profitable_values.median()
    unprofitable_median = unprofitable_values.median()

    rank_biserial = (
        2 * test_result.statistic
        / (
            profitable_n
            * unprofitable_n
        )
        - 1
    )

    return {
        "feature": feature,
        "profitable_n": profitable_n,
        "unprofitable_n": unprofitable_n,
        "profitable_median": profitable_median,
        "unprofitable_median": unprofitable_median,
        "median_difference": (
            profitable_median
            - unprofitable_median
        ),
        "u_statistic": test_result.statistic,
        "p_value": test_result.pvalue,
        "rank_biserial": rank_biserial,
    }

# Hypothesis 1: Trade/idea holding duration

## Max holding duration of losing to profitable position

In [109]:
# Calculate each trade's holding duration.
trades_with_duration = trades.copy()

trades_with_duration[
    "trade_duration_minutes"
] = (
    trades_with_duration["close_date_time"]
    - trades_with_duration["open_date_time"]
).dt.total_seconds() / 60

trades_with_duration[
    "profitable_trade_duration_minutes"
] = (
    trades_with_duration[
        "trade_duration_minutes"
    ]
    .where(
        trades_with_duration["net_profit"] > 0
    )
)

trades_with_duration[
    "losing_trade_duration_minutes"
] = (
    trades_with_duration[
        "trade_duration_minutes"
    ]
    .where(
        trades_with_duration["net_profit"] < 0
    )
)

maximum_trade_duration_features = (
    trades_with_duration
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        maximum_profitable_trade_duration_minutes=(
            "profitable_trade_duration_minutes",
            "max",
        ),
        maximum_losing_trade_duration_minutes=(
            "losing_trade_duration_minutes",
            "max",
        ),
        profitable_trade_count=(
            "profitable_trade_duration_minutes",
            "count",
        ),
        losing_trade_count=(
            "losing_trade_duration_minutes",
            "count",
        ),
    )
)

valid_duration_ratio = (
    maximum_trade_duration_features[
        "maximum_profitable_trade_duration_minutes"
    ].gt(0)
    &
    maximum_trade_duration_features[
        "maximum_losing_trade_duration_minutes"
    ].gt(0)
)

maximum_trade_duration_features[
    "maximum_losing_to_profitable_trade_duration_ratio"
] = np.where(
    valid_duration_ratio,
    (
        maximum_trade_duration_features[
            "maximum_losing_trade_duration_minutes"
        ]
        / maximum_trade_duration_features[
            "maximum_profitable_trade_duration_minutes"
        ]
    ),
    np.nan,
)

maximum_trade_duration_features[
    "log_maximum_losing_to_profitable_trade_duration_ratio"
] = np.log(
    maximum_trade_duration_features[
        "maximum_losing_to_profitable_trade_duration_ratio"
    ]
)

feature_columns = [
    "log_maximum_losing_to_profitable_trade_duration_ratio",
]

maximum_trade_duration_features[
    feature_columns
] = (
    maximum_trade_duration_features[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

maximum_trade_duration_features = (
    maximum_trade_duration_features
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)

assert not maximum_trade_duration_features[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

assert (
    maximum_trade_duration_features[
        "profitable_trade_count"
    ] >= 1
).all()

assert (
    maximum_trade_duration_features[
        "losing_trade_count"
    ] >= 1
).all()

test_data = (
    account_campaign_outcomes[
        [
            "account_id",
            "campaign_id",
            "account_campaign_net_profit",
            "profitable",
            "profitability_group",
        ]
    ]
    .merge(
        maximum_trade_duration_features,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert not test_data[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

test_results = pd.DataFrame(
    [
        run_mann_whitney_comparison(
            data=test_data,
            feature=feature,
        )
        for feature in feature_columns
    ]
)

alpha = 0.05
adjusted_alpha = alpha / len(feature_columns)

test_results["adjusted_alpha"] = adjusted_alpha

test_results["reject_null"] = (
    test_results["p_value"]
    < test_results["adjusted_alpha"]
)

feature_labels = {
    (
        "log_maximum_losing_to_profitable_"
        "trade_duration_ratio"
    ): (
        "Log maximum losing-to-profitable "
        "trade duration ratio"
    ),
}

test_results["feature"] = (
    test_results["feature"]
    .replace(feature_labels)
)

test_display = (
    test_results[
        [
            "feature",
            "profitable_n",
            "unprofitable_n",
            "profitable_median",
            "unprofitable_median",
            "median_difference",
            "u_statistic",
            "p_value",
            "adjusted_alpha",
            "rank_biserial",
            "reject_null",
        ]
    ]
    .copy()
)

for column in [
    "profitable_median",
    "unprofitable_median",
    "median_difference",
]:
    test_display[column] = (
        test_display[column]
        .round(4)
    )

test_display["u_statistic"] = (
    test_display["u_statistic"]
    .round(0)
)

test_display["p_value"] = (
    test_display["p_value"]
    .map(
        lambda value: (
            f"{value:.3e}"
            if pd.notna(value)
            else "NaN"
        )
    )
)

test_display["adjusted_alpha"] = (
    test_display["adjusted_alpha"]
    .round(3)
)

test_display["rank_biserial"] = (
    test_display["rank_biserial"]
    .round(3)
)

test_display = (
    test_display
    .set_index("feature")
    .T
)

test_display.index.name = None
test_display.columns.name = None

print(
    "Account-campaigns included:",
    len(test_data),
)

print(
    "Adjusted alpha:",
    adjusted_alpha,
)

display(test_display)

Account-campaigns included: 4651
Adjusted alpha: 0.05


,Log maximum losing-to-profitable trade duration ratio
profitable_n,1615
unprofitable_n,3036
profitable_median,-0.9931
unprofitable_median,0.5262
median_difference,-1.5192
u_statistic,1067721.0
p_value,4.211e-221
adjusted_alpha,0.05
rank_biserial,-0.564
reject_null,True


In [110]:
duration_trade_test_results = test_results.copy()

## Max holding duration of losing to profitable idea

In [111]:
# Build maximum idea-duration features.
maximum_idea_duration_features = (
    idea_features
    .assign(
        profitable_idea_duration_minutes=lambda data: (
            data["idea_duration_minutes"]
            .where(data["is_profitable_idea"])
        ),
        losing_idea_duration_minutes=lambda data: (
            data["idea_duration_minutes"]
            .where(data["is_losing_idea"])
        ),
    )
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        maximum_profitable_idea_duration_minutes=(
            "profitable_idea_duration_minutes",
            "max",
        ),
        maximum_losing_idea_duration_minutes=(
            "losing_idea_duration_minutes",
            "max",
        ),
        profitable_idea_count=(
            "profitable_idea_duration_minutes",
            "count",
        ),
        losing_idea_count=(
            "losing_idea_duration_minutes",
            "count",
        ),
    )
)

valid_duration_ratio = (
    maximum_idea_duration_features[
        "maximum_profitable_idea_duration_minutes"
    ].gt(0)
    &
    maximum_idea_duration_features[
        "maximum_losing_idea_duration_minutes"
    ].gt(0)
)

maximum_idea_duration_features[
    "maximum_losing_to_profitable_idea_duration_ratio"
] = np.where(
    valid_duration_ratio,
    (
        maximum_idea_duration_features[
            "maximum_losing_idea_duration_minutes"
        ]
        / maximum_idea_duration_features[
            "maximum_profitable_idea_duration_minutes"
        ]
    ),
    np.nan,
)

maximum_idea_duration_features[
    "log_maximum_losing_to_profitable_idea_duration_ratio"
] = np.log(
    maximum_idea_duration_features[
        "maximum_losing_to_profitable_idea_duration_ratio"
    ]
)

feature_columns = [
    "log_maximum_losing_to_profitable_idea_duration_ratio",
]

maximum_idea_duration_features[
    feature_columns
] = (
    maximum_idea_duration_features[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)

maximum_idea_duration_features = (
    maximum_idea_duration_features
    .dropna(
        subset=feature_columns
    )
    .reset_index(drop=True)
)

assert not maximum_idea_duration_features[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

assert (
    maximum_idea_duration_features[
        "profitable_idea_count"
    ] >= 1
).all()

assert (
    maximum_idea_duration_features[
        "losing_idea_count"
    ] >= 1
).all()

test_data = (
    account_campaign_outcomes[
        [
            "account_id",
            "campaign_id",
            "account_campaign_net_profit",
            "profitable",
            "profitability_group",
        ]
    ]
    .merge(
        maximum_idea_duration_features,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert not test_data[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

test_results = pd.DataFrame(
    [
        run_mann_whitney_comparison(
            data=test_data,
            feature=feature,
        )
        for feature in feature_columns
    ]
)

alpha = 0.05
adjusted_alpha = alpha / len(feature_columns)

test_results["adjusted_alpha"] = adjusted_alpha

test_results["reject_null"] = (
    test_results["p_value"]
    < test_results["adjusted_alpha"]
)

feature_labels = {
    (
        "log_maximum_losing_to_profitable_"
        "idea_duration_ratio"
    ): (
        "Log maximum losing-to-profitable "
        "idea duration ratio"
    ),
}

test_results["feature"] = (
    test_results["feature"]
    .replace(feature_labels)
)

test_display = (
    test_results[
        [
            "feature",
            "profitable_n",
            "unprofitable_n",
            "profitable_median",
            "unprofitable_median",
            "median_difference",
            "u_statistic",
            "p_value",
            "adjusted_alpha",
            "rank_biserial",
            "reject_null",
        ]
    ]
    .copy()
)

for column in [
    "profitable_median",
    "unprofitable_median",
    "median_difference",
]:
    test_display[column] = (
        test_display[column]
        .round(4)
    )

test_display["u_statistic"] = (
    test_display["u_statistic"]
    .round(0)
)

test_display["p_value"] = (
    test_display["p_value"]
    .map(
        lambda value: (
            f"{value:.3e}"
            if pd.notna(value)
            else "NaN"
        )
    )
)

test_display["adjusted_alpha"] = (
    test_display["adjusted_alpha"]
    .round(3)
)

test_display["rank_biserial"] = (
    test_display["rank_biserial"]
    .round(3)
)

test_display = (
    test_display
    .set_index("feature")
    .T
)

test_display.index.name = None
test_display.columns.name = None

print(
    "Account-campaigns included:",
    len(test_data),
)

print(
    "Adjusted alpha:",
    adjusted_alpha,
)

display(test_display)

Account-campaigns included: 3942
Adjusted alpha: 0.05


,Log maximum losing-to-profitable idea duration ratio
profitable_n,1372
unprofitable_n,2570
profitable_median,-0.8585
unprofitable_median,0.4981
median_difference,-1.3566
u_statistic,805104.0
p_value,2.970e-174
adjusted_alpha,0.05
rank_biserial,-0.543
reject_null,True


In [112]:
duration_idea_test_results = test_results.copy()

# Hypothesis 2: Trading intensity

In [113]:
# Build account-campaign trading activity features.
account_campaign_trade_activity = (
    trades
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        first_trade_open_time=(
            "open_date_time",
            "min",
        ),
        last_trade_close_time=(
            "close_date_time",
            "max",
        ),
        total_trades=(
            "account_id",
            "size",
        ),
    )
)

account_campaign_trade_activity[
    "active_duration_hours"
] = (
    account_campaign_trade_activity[
        "last_trade_close_time"
    ]
    - account_campaign_trade_activity[
        "first_trade_open_time"
    ]
).dt.total_seconds() / 3600

account_campaign_trade_activity.loc[
    account_campaign_trade_activity[
        "active_duration_hours"
    ] <= 0,
    "active_duration_hours",
] = np.nan


# Build account-campaign idea counts.
account_campaign_idea_activity = (
    idea_features
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        total_ideas=(
            "idea_id",
            "nunique",
        ),
    )
)


# Combine the trade and idea activity features.
trading_intensity_features = (
    account_campaign_trade_activity
    .merge(
        account_campaign_idea_activity,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="left",
        validate="one_to_one",
    )
)

trading_intensity_features[
    "trades_per_active_hour"
] = (
    trading_intensity_features["total_trades"]
    / trading_intensity_features["active_duration_hours"]
)

trading_intensity_features[
    "ideas_per_active_hour"
] = (
    trading_intensity_features["total_ideas"]
    / trading_intensity_features["active_duration_hours"]
)


# Specify the features to test.
feature_columns = [
    "trades_per_active_hour",
    "ideas_per_active_hour",
]

trading_intensity_features[
    feature_columns
] = (
    trading_intensity_features[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


# Validate that there is one row per account-campaign.
assert not trading_intensity_features[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

assert (
    trading_intensity_features["total_trades"] >= 1
).all()

assert (
    trading_intensity_features["total_ideas"] >= 1
).all()


# Merge the behavioural features with the reusable outcome table.
test_data = (
    account_campaign_outcomes[
        [
            "account_id",
            "campaign_id",
            "account_campaign_net_profit",
            "profitable",
            "profitability_group",
        ]
    ]
    .merge(
        trading_intensity_features,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="inner",
        validate="one_to_one",
    )
)

assert not test_data[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()


# Run a Mann–Whitney U test for each feature.
test_results = pd.DataFrame(
    [
        run_mann_whitney_comparison(
            data=test_data,
            feature=feature,
        )
        for feature in feature_columns
    ]
)


# Apply Bonferroni correction based on the number of tested features.
alpha = 0.05

adjusted_alpha = (
    alpha
    / len(feature_columns)
)

test_results[
    "adjusted_alpha"
] = adjusted_alpha

test_results[
    "reject_null"
] = (
    test_results["p_value"]
    < test_results["adjusted_alpha"]
)


# Add readable feature labels.
feature_labels = {
    "trades_per_active_hour": (
        "Trades per active hour"
    ),
    "ideas_per_active_hour": (
        "Ideas per active hour"
    ),
}

test_results[
    "feature"
] = (
    test_results["feature"]
    .replace(feature_labels)
)


# Format the test results.
test_display = (
    test_results[
        [
            "feature",
            "profitable_n",
            "unprofitable_n",
            "profitable_median",
            "unprofitable_median",
            "median_difference",
            "u_statistic",
            "p_value",
            "adjusted_alpha",
            "rank_biserial",
            "reject_null",
        ]
    ]
    .copy()
)

for column in [
    "profitable_median",
    "unprofitable_median",
    "median_difference",
]:
    test_display[column] = (
        test_display[column]
        .round(4)
    )

test_display["u_statistic"] = (
    test_display["u_statistic"]
    .round(0)
)

test_display["p_value"] = (
    test_display["p_value"]
    .map(
        lambda value: (
            f"{value:.3e}"
            if pd.notna(value)
            else "NaN"
        )
    )
)

test_display["adjusted_alpha"] = (
    test_display["adjusted_alpha"]
    .round(3)
)

test_display["rank_biserial"] = (
    test_display["rank_biserial"]
    .round(3)
)

test_display = (
    test_display
    .set_index("feature")
    .T
)

test_display.index.name = None
test_display.columns.name = None

print(
    "Account-campaigns included:",
    len(test_data),
)

print(
    "Adjusted alpha:",
    adjusted_alpha,
)

display(test_display)

Account-campaigns included: 8165
Adjusted alpha: 0.025


,Trades per active hour,Ideas per active hour
profitable_n,2703,2703
unprofitable_n,5462,5462
profitable_median,1.3168,1.0234
unprofitable_median,2.279,1.7678
median_difference,-0.9623,-0.7444
u_statistic,5974921.0,5901162.0
p_value,9.263e-45,2.194e-49
adjusted_alpha,0.025,0.025
rank_biserial,-0.191,-0.201
reject_null,True,True


In [114]:
trading_intensity_test_results = test_results.copy()

# Hypothesis 3: Direction-switching

## Direction-switching waiting time

In [115]:
# Arrange ideas chronologically within each account, campaign
idea_sequence = (
    idea_features[
        [
            "account_id",
            "campaign_id",
            "idea_id",
            "side",
            "idea_start_time",
            "idea_end_time",
        ]
    ]
    .sort_values(
        [
            "account_id",
            "campaign_id",
            "idea_start_time",
            "idea_end_time",
            "idea_id",
        ]
    )
    .copy()
)


# Define the groups within which consecutive ideas are compared.
sequence_group_columns = [
    "account_id",
    "campaign_id",
]


# Attach information from the previous idea.
idea_sequence["previous_idea_id"] = (
    idea_sequence
    .groupby(
        sequence_group_columns,
        sort=False,
    )["idea_id"]
    .shift(1)
)

idea_sequence["previous_side"] = (
    idea_sequence
    .groupby(
        sequence_group_columns,
        sort=False,
    )["side"]
    .shift(1)
)

idea_sequence["previous_idea_end_time"] = (
    idea_sequence
    .groupby(
        sequence_group_columns,
        sort=False,
    )["idea_end_time"]
    .shift(1)
)


# Identify transitions between consecutive ideas.
idea_sequence["eligible_transition"] = (
    idea_sequence["previous_idea_id"].notna()
)

idea_sequence["direction_switch"] = (
    idea_sequence["eligible_transition"]
    & (
        idea_sequence["side"]
        != idea_sequence["previous_side"]
    )
)


# Calculate the waiting time from the previous idea's end
# to the current idea's start.
idea_sequence[
    "previous_to_current_gap_minutes"
] = (
    idea_sequence["idea_start_time"]
    - idea_sequence["previous_idea_end_time"]
).dt.total_seconds() / 60


# Keep only non-overlapping transitions.
idea_sequence["non_overlapping_transition"] = (
    idea_sequence["eligible_transition"]
    & (
        idea_sequence[
            "previous_to_current_gap_minutes"
        ] >= 0
    )
)


# Calculate the median waiting time after a direction switch
# for each account-campaign.
direction_switch_wait_features = (
    idea_sequence.loc[
        idea_sequence["direction_switch"]
        & idea_sequence["non_overlapping_transition"]
    ]
    .groupby(
        [
            "account_id",
            "campaign_id",
        ],
        as_index=False,
    )
    .agg(
        median_direction_switch_gap_minutes=(
            "previous_to_current_gap_minutes",
            "median",
        ),
        direction_switch_gap_count=(
            "previous_to_current_gap_minutes",
            "count",
        ),
    )
)


# Specify the feature to test.
feature_columns = [
    "median_direction_switch_gap_minutes",
]


# Replace invalid values.
direction_switch_wait_features[
    feature_columns
] = (
    direction_switch_wait_features[
        feature_columns
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
)


# Validate that there is one row per account-campaign.
assert not direction_switch_wait_features[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()

assert (
    direction_switch_wait_features[
        "direction_switch_gap_count"
    ] >= 1
).all()


# Merge the behavioural feature with the reusable outcome table.
test_data = (
    account_campaign_outcomes[
        [
            "account_id",
            "campaign_id",
            "account_campaign_net_profit",
            "profitable",
            "profitability_group",
        ]
    ]
    .merge(
        direction_switch_wait_features,
        on=[
            "account_id",
            "campaign_id",
        ],
        how="inner",
        validate="one_to_one",
    )
)


# Confirm that each tested observation is one account-campaign.
assert not test_data[
    [
        "account_id",
        "campaign_id",
    ]
].duplicated().any()


# Run the Mann–Whitney U test.
test_results = pd.DataFrame(
    [
        run_mann_whitney_comparison(
            data=test_data,
            feature=feature,
        )
        for feature in feature_columns
    ]
)


# Apply the significance threshold.
alpha = 0.05

adjusted_alpha = (
    alpha
    / len(feature_columns)
)

test_results["adjusted_alpha"] = adjusted_alpha

test_results["reject_null"] = (
    test_results["p_value"]
    < test_results["adjusted_alpha"]
)


# Add a readable feature label.
feature_labels = {
    "median_direction_switch_gap_minutes": (
        "Median direction-switch waiting time"
    ),
}

test_results["feature"] = (
    test_results["feature"]
    .replace(feature_labels)
)


# Format the test results.
test_display = (
    test_results[
        [
            "feature",
            "profitable_n",
            "unprofitable_n",
            "profitable_median",
            "unprofitable_median",
            "median_difference",
            "u_statistic",
            "p_value",
            "adjusted_alpha",
            "rank_biserial",
            "reject_null",
        ]
    ]
    .copy()
)

for column in [
    "profitable_median",
    "unprofitable_median",
    "median_difference",
]:
    test_display[column] = (
        test_display[column]
        .round(4)
    )

test_display["u_statistic"] = (
    test_display["u_statistic"]
    .round(0)
)

test_display["p_value"] = (
    test_display["p_value"]
    .map(
        lambda value: (
            f"{value:.3e}"
            if pd.notna(value)
            else "NaN"
        )
    )
)

test_display["adjusted_alpha"] = (
    test_display["adjusted_alpha"]
    .round(3)
)

test_display["rank_biserial"] = (
    test_display["rank_biserial"]
    .round(3)
)

test_display = (
    test_display
    .set_index("feature")
    .T
)

test_display.index.name = None
test_display.columns.name = None


print(
    "Account-campaigns included:",
    len(test_data),
)

print(
    "Adjusted alpha:",
    adjusted_alpha,
)

display(test_display)

Account-campaigns included: 4517
Adjusted alpha: 0.05


,Median direction-switch waiting time
profitable_n,1671
unprofitable_n,2846
profitable_median,6.6
unprofitable_median,3.8208
median_difference,2.7792
u_statistic,2709388.0
p_value,4.668e-15
adjusted_alpha,0.05
rank_biserial,0.139
reject_null,True


In [116]:
direction_switch_test_results = test_results.copy()

# Save outputs

In [117]:
stage1_hypothesis_results = pd.concat(
    [
        duration_idea_test_results.assign(
            hypothesis="Maximum idea holding duration",
            analysis_level="Idea",
        ),
        duration_trade_test_results.assign(
            hypothesis="Maximum trade holding duration",
            analysis_level="Trade",
        ),
        trading_intensity_test_results.assign(
            hypothesis="Trading intensity",
            analysis_level="Account-campaign",
        ),
        direction_switch_test_results.assign(
            hypothesis="Direction-switching waiting time",
            analysis_level="Account-campaign",
        ),
    ],
    ignore_index=True,
)

In [118]:
from pathlib import Path

STAGE1_OUTPUT_DIR = Path("outputs/stage1")
STAGE1_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


dataframe_exports = {
    "trader.parquet": trader,
    "trades.parquet": trades,
    "trades_with_ideas.parquet": trades_with_ideas,
    (
        "trades_with_previous_completed.parquet"
    ): trades_with_previous_completed,
    "idea_features_stage1.parquet": idea_features,
    (
        "account_campaign_outcomes_stage1.parquet"
    ): account_campaign_outcomes,
}


for file_name, dataframe in dataframe_exports.items():
    dataframe.to_parquet(
        STAGE1_OUTPUT_DIR / file_name,
        index=False,
    )


report_exports = {
    "raw_file_audit.csv": audit_df,
    "campaign_summary.csv": campaign_summary,
    (
        "trader_loading_report.csv"
    ): trader_loading_report_df,
    (
        "trade_loading_report.csv"
    ): trade_loading_report_df,
    (
        "stage1_hypothesis_results.csv"
    ): stage1_hypothesis_results,
}


for file_name, dataframe in report_exports.items():
    dataframe.to_csv(
        STAGE1_OUTPUT_DIR / file_name,
        index=False,
    )


print("Saved Stage 1 outputs:")

for output_file in sorted(STAGE1_OUTPUT_DIR.iterdir()):
    print(
        f"- {output_file.name}"
    )

print(
    "\nOutput directory:",
    STAGE1_OUTPUT_DIR.resolve(),
)

Saved Stage 1 outputs:
- account_campaign_outcomes_stage1.parquet
- campaign_summary.csv
- idea_features_stage1.parquet
- raw_file_audit.csv
- stage1_hypothesis_results.csv
- trade_loading_report.csv
- trader.parquet
- trader_loading_report.csv
- trades.parquet
- trades_with_ideas.parquet
- trades_with_previous_completed.parquet

Output directory: C:\Desktop\C22-veNTUre\outputs\stage1
